# Detect Suspicious Value Transfers in Poker

**Public LB: 0.80534** · CPU · pipeline `poker_baseline_080.py` · reuse `step3` cache (pair-hand features rebuild once)

This notebook is the write-up plus runnable code for the **0.80534** line, plus one change: a population **postflop continue policy**. The first half explains the algorithm, metrics, features, and formulas. The second half is the full pipeline (Polars lazy + LightGBM + family evidence specialists).

| Public | Meaning |
|---|---|
| 0.69536 | chip-flow / sequence templates (`poker140926`) |
| 0.68796 / 0.68653 / 0.68624 | regenerated PU cache; structural extras on the 0.69 path |
| **0.80136** | card-conditional **preflop** policy + all eligible pairs + two-step PU |
| **0.80534** | + family evidence specialists (`0.60` spec + `0.40` global) |
| this notebook | + $P(\mathrm{continue}\mid \mathrm{cat},\mathrm{street})$ and junk-call-after-partner-raise |

Keep the same booster and the same pair index. The jump 0.69 → 0.80 happened because **features sit on the generator's policy**, not because trees got deeper. Specialists moved MAP@5 only (`hs_*` stays on the global ranker). Postflop policy is the next Pair-AP lever.

**Contents**

1. Problem, PU labels, and metric
2. Algorithm — two outputs, two feature spaces
3. Workflow
4. Feature list
5. Hand-level formulas
6. Pair-level formulas
7. Experiments not to repeat
8. Runnable pipeline


## 1. Problem, PU labels, and metric

The dataset is **synthetic 6-max NLHE**: ~2M hands, ~12k players, **400 pools × 30** with no mixing (`table_id`). Time split 60/40 by `hands.phase` (`development` / `evaluation`). All six hole cards and the board are visible, so card-conditional policy is measurable on every seat.

Each submission row is a **player pair** on eval (112,540 pairs):

| Column | Role | Weight |
|---|---|---|
| `risk_score` $\in [0,1]$ | collusion or not | 70% Pair AP |
| `predicted_behavior` | `none` or one of 3 published families | 10% Behavior MAP |
| `evidence_hand_1..5` | hands that include **both** players, or `NO_EVIDENCE` | 20% MAP@5 |

$$
\mathrm{Score} = 0.70 \cdot \mathrm{PairAP} + 0.20 \cdot \mathrm{MAP@5} + 0.10 \cdot \mathrm{BehaviorMAP}
$$

`other_coordination` is valid on the form but **does not enter Behavior MAP** — never predict that class. The host breaks `risk_score` ties by `pair_id`, so scores must be unique.

### Labels are PU, not PN

`development_labels.csv`: 1,860 confirmed pairs (372 positive / 1,488 confirmed-negative). Every co-table pair **not listed is unlabeled, not negative**. Training only on the 1,860 labeled pairs yields labelled OOF AP ~0.97 while public Pair AP collapses: the model never sees the “normal” pairs that dominate the 112k eval set.

This pipeline does **not** sample 60 unlabeled pairs per table. Eligible development pairs are every co-seated pair with at least 57 shared hands (same 1.9% rule as eval 38/2000) that does not contain a publicly-positive player — on the order of **137k** pairs. Track **eval-mirrored AP** (treat unlabeled as negative on that full population). Do not pick models with labelled AP.

### MAP@5 (evidence)

For pair $p$, $R_p$ is the planted-hand set. The model returns an order $h_1,\ldots,h_5$. Let $\mathrm{hits}(k)$ be the number of relevant items in the first $k$ positions **after** $h_k$ is a hit:

$$
\mathrm{AP@5}(p)=\frac{1}{\min(|R_p|,5)}\sum_{k=1}^{5}\mathbf{1}[h_k\in R_p]\cdot\frac{\mathrm{hits}(k)}{k}
$$

Official: planted evidence is **specific behavior visible in `actions.parquet`**. Hidden chip flow alone is never evidence. Coordination is episodic — so evidence is **ranking hands within a pair**, not classifying the pair. Always fill five slots; eval pairs have $\ge 38$ shared hands, so `NO_EVIDENCE` should not appear.

### Behavior MAP

Only pairs marked collusive (`risk` in the top rate $\rho$) carry a family. Macro OvR AP over 3 classes. Isolation is the weakest family on MAP@5, which is why postflop policy (fold a made hand to the partner; junk-call after a partner raise) is in this revision.


## 2. Algorithm — two outputs, two feature spaces

The planted signal is **not chip flow**. The simulator's policy is strictly card-driven (VPIP $\approx 2\%$ at Chen $-1.5$, $\approx 96\%$ at Chen $\ge 12$), so two partners entering the same pot with junk hole cards is a large joint policy deviation. Soft play and isolation live **after the flop**: check with two-pair-plus, fold a made hand to the partner, call air after the partner bets.

The system splits **pair risk** from **hand evidence**. On the same hand row, pair-level `hs_*` may ignore the specialist score.

```
actions / seats / hands / board
        │
        ▼
 player × hand + policy + 7-card strength
        │
        ▼
 pair × hand features  ──┬── aggregate ──► pair X ──► risk + family
                         │                  (hs_* = global ranker)
                         └── within-pair ranks ──► evidence MAP@5
                                                (0.60 spec + 0.40 global)
```

**Channel-split principle** (this is what won 0.80136 → 0.80534):

1. **Pair X** describes the pair's *long-run habits*: policy-surprise rates, tails, partner-vs-field, burst. `hs_*` comes from the **global** hand ranker so Pair AP does not move when MAP@5 mix changes.
2. **Evidence X** is the same hand row, scored by a global ranker **and** three family specialists. Slots use a **fixed** mix $0.60\,\sum_k w_k\,p_{\mathrm{spec},k}+0.40\,p_{\mathrm{glob}}$, where $w$ is the row-normalized family detector (same scores as $\arg\max$). `none` still uses the global ranker only. Do not grid-search $0.60/0.40$ on 372 labelled pairs.
3. Putting specialist scores into pair `hs_*` would confound Pair AP with family accuracy. Don't.

### 2.1 Pair risk

One LightGBM bag on the full eligible population, **two-step PU**:

- Stage 1: unlabeled weight $0.1$. Thresholds from the labelled-positive OOF: $q_{0.05}$ (ambiguous) and $q_{0.50}$ (pseudo-positive).
- Stage 2: pseudo-positives get label $1$ weight $1.0$; ambiguous get weight $0$; remaining unlabeled stay $0.1$.

Three LightGBM one-vs-rest family detectors (`num_leaves=7`). Family $= \arg\max$ of the three scores, divided by the labelled prior. Positive rate $\rho$ is chosen on a fixed grid; take the **largest rate $\le 5\%$** still within $0.0005$ of the best development host metric. Pairs outside the top-$\rho$ get `none`.

Sample weight at fit:

$$
w_i=\begin{cases}
5.0 & y_i=1\text{ labelled}\\
1.0 & \text{confirmed negative}\\
1.0 & \text{stage-2 pseudo-positive}\\
0.1 & \text{reliable unlabeled}\\
0 & \text{ambiguous unlabeled}
\end{cases}
$$

CV: tables shuffled into 5 folds — the 30 players in a pool must not leak into another fold.

Gate for this revision: **eval-mirrored Pair AP**, not labelled AP.

### 2.2 Evidence

Global: LightGBM bag on all positive-pair hands, target = planted evidence hand.

By family: one bag per `{directed_transfer, soft_play, coordinated_isolation}` if that family has $\ge 200$ train rows, else fall back to the global bag.

Final blend is **not z-scored**:

$$
s_{\mathrm{ev}}=0.60\,p_{\mathrm{spec}}(\hat f)+0.40\,p_{\mathrm{glob}}
$$

$\hat f$ is the pair's **predicted family** (not the label — eval has no labels). `none` / unknown → global only. Score every 112,540 pairs; always emit five evidence hands, unique `risk_score`.


## 3. Workflow

```mermaid
flowchart TD
    A[Raw parquet + labels] --> B[Action context: last_aggr]
    B --> C[Player × hand: Chen, 169-class, VPIP/PFR]
    C --> D[Preflop policy P vpip]
    D --> E[7-card strength + rank among six]
    E --> F[Postflop policy P continue]
    C --> G[Pair index: all eligible ≥ 57 shared]
    F --> H[Pair × hand features]
    G --> H
    H --> I[Global hand ranker + 3 family specialists]
    I --> J[Pair aggregate — hs from global]
    J --> K[Two-step PU pair model + family]
    I --> L["Evidence 0.60 spec + 0.40 global"]
    K --> M[submission.csv]
    L --> M
```

1. **Action context**: `is_aggr`, `last_aggr` forward-fill by street, sizes in BB.
2. **Player-hand**: position, Chen, 169-class hole, VPIP/PFR, action counts.
3. **Preflop policy**: $P(\mathrm{vpip}\mid \mathrm{class},\mathrm{pos},\mathrm{faced\text{-}raise})$ on all 12M seats, Laplace $k=20$.
4. **Exact strength**: best 5-of-5/6/7 at flop/turn/river; rank among the six dealt players.
5. **Postflop policy** *(this revision)*: $P(\mathrm{continue}\mid \mathrm{cat},\mathrm{street})$ on every postflop faced-bet action.
6. **Pair index**: every eligible development pair; no 60-per-table sample.
7. **Pair-hand**: joint surprise, partner-street tells, junk-call-after-partner-raise, both-passive-made.
8. **Fit** hand models → **aggregate** with global `hand_score` → **fit** two-step PU → **infer** eval, write `/kaggle/working/submission.csv`.

Reuse `action_context` / `player_hands` / preflop policy / strength / pair index from `/kaggle/working/step3`. Pair-hand features and later stages rebuild if the new columns are missing. Cold run ~16 min on 10 cores; warm cache ~4 min; first run of this revision spends extra on pair-hand rebuild.


## 4. Feature list

The base grain is **one row = one `(pair_id, hand_id)`**. Pair X is statistics over those rows. Evidence uses the same row plus within-pair ranks (`_prank`, `_to_max`).

### 4.1 Player × hand (before pairing)

| Group | Column | Meaning |
|---|---|---|
| Position / hole | `pos`, `hole_class`, `chen` | 0=BTN … 5=CO; 169-class; Chen prior |
| VPIP | `vpip`, `vpip_a`, `pfr`, `pf_faced_raise`, `pf_entry_no` | contribution vs action-defined entry |
| Preflop policy | `p_vpip`, `loose`, `tight`, `junk_vpip`, `vpip_resid` | surprise vs $P(\mathrm{vpip}\mid c,\mathrm{pos},\mathrm{raise})$ |
| Postflop policy | `post_loose`, `post_tight`, `post_resid`, `junk_continue`, `fold_made` | surprise vs $P(\mathrm{continue}\mid \mathrm{cat},\mathrm{street})$ |
| Strength | `v_flop/turn/river/final`, `cat_final`, `rk_*` | exact 7-card value and rank among six |
| Action counts | `n_aggr`, `n_call`, `n_fold_facing`, `n_postflop_check`, … | frequencies |

### 4.2 Pair × hand — policy surprise

| Column | Meaning |
|---|---|
| `loose_sum`, `loose_min`, `junk_vpip_sum` | joint preflop surprise |
| `second_entrant_junk`, `both_vpip_junk`, `both_surprising` | both enter, second/both with junk |
| `post_loose_sum`, `post_loose_min`, `post_loose_partner` | postflop continue-surprise, all bets vs partner bets |
| `junk_continue_sum`, `junk_call_partner_post` | continued with high-card / one-pair |
| `fold_better_surprise`, `fold_made_partner` | folded two-pair-plus, especially to the partner |
| `made_check_sum`, `made_check_hu` | checked a made hand (soft-play grain) |

### 4.3 Pair × hand — templates

| Column | Template |
|---|---|
| `junk_call_after_partner_raise` | call with $\mathrm{cat}\le 1$ facing the partner after the flop |
| `both_passive_made` | both $\mathrm{cat\_final}\ge 2$, no aggression, both show down |
| `fold_made_to_partner` | folded two-pair-plus to the partner |
| `fold_better_to_partner`, `dump_better_hand` | stronger hand dumped / folded to partner |
| `hu_check_two_pair_plus`, `checkdown` | HU check-down with a made hand |
| `pf_squeeze`, `squeeze_then_fold_pf` | both aggress + outsider folds, then one folds to partner |
| `transfer_any`, `partner_won_other_lost` | chip moved between the two in this hand |

### 4.4 Pair X — long-run statistics

From each hand col $c$: $c_{\mathrm{mean}}$, and for the tail $c_{\mathrm{max}}$, $c_{\mathrm{top3}}$, $c_{\mathrm{sum}}$ (lists `AGG_MEAN / AGG_MAX / AGG_TOP3 / AGG_SUM`).

Plus `hs_mean`, `hs_top3`, `hs_top5`, `hs_n95` from the **global** hand score; 8-bin phase burst; transfer imbalance / rate / dominant side.

Each metric in `TRANK_COLS` also gets `_trank` = rank at the table / number of table pairs.

### 4.5 Partner vs field

For player $i$ in the pair, compare expected behavior **when sitting with the partner** vs **when the partner is absent** (same phase), shrunk by $n/(n+25)$. Contrast columns include `loose`, `junk_vpip`, `post_loose`, `junk_continue`. Then sum / gap / min of the two sides.


## 5. Hand-level formulas

Notation: $n_1,n_2$ = `net_bb`; $BB$ = big blind. Position $= (\texttt{seat\_no}-\texttt{button\_seat}+6)\bmod 6$.

### Chen (preflop prior)

Standard Chen formula on ranks $r\in\{2,\ldots,14\}$. Used as the preflop policy prior and for `junk_vpip` $= \mathbf{1}[\mathrm{vpip}]\cdot(8-\mathrm{Chen})_+$.

### 169-class hole

$$
\mathrm{class}=15\cdot\max(r_1,r_2)+\min(r_1,r_2)+300\cdot\mathbf{1}[\text{suited}]
$$

### Preflop policy

On all 12M seats, Laplace-smoothed toward the class mean ($k=20$):

$$
\hat P(\mathrm{vpip}\mid c,\mathrm{pos},\mathrm{raise})=\mathrm{clip}\!\left(\frac{s+k\,\bar p_c}{n+k},\,10^{-4},\,1-10^{-4}\right)
$$

$$
\texttt{loose}=\mathbf{1}[\mathrm{vpip}]\cdot\bigl(-\log\hat P\bigr),\qquad
\texttt{tight}=\mathbf{1}[\neg\mathrm{vpip}]\cdot\bigl(-\log(1-\hat P)\bigr)
$$

Junk entries are large because $\hat P$ is tiny at low Chen / bad class.

### Exact strength

5-card value $=\mathrm{category}\cdot 15^5 + \text{tie-break}$, category $0$ = high card … $8$ = straight flush. Best 5-of-5/6/7 at flop/turn/river. Rank among the six dealt players (not among those who saw the street).

### Postflop policy *(this revision)*

Every action with $\texttt{street}>0$ and $\texttt{to\_call}>0$:

$$
\mathrm{cat}=\bigl\lfloor v_{\mathrm{street}}/15^5\bigr\rfloor,\qquad
\mathrm{continue}=\mathbf{1}[\texttt{action}\ne\texttt{fold}]
$$

$$
\hat P(\mathrm{continue}\mid \mathrm{cat},\mathrm{street})=\mathrm{clip}\!\left(\frac{s+k\,\bar p_{\mathrm{cat}}}{n+k},\,10^{-4},\,1-10^{-4}\right)
$$

$$
\texttt{post\_loose}=\mathbf{1}[\mathrm{continue}]\cdot\bigl(-\log\hat P\bigr)
$$

Honest players fold air and continue with made hands. Continuing high-card / one-pair ($\mathrm{cat}\le 1$) after a **partner** bet, or folding two-pair-plus ($\mathrm{cat}\ge 2$) to the partner, is the postflop analogue of junk VPIP. The population table does **not** condition on partner — that deviation is measured at pair-hand time by applying the same $\hat P$ on `facing_partner` rows.

$$
\texttt{junk\_call\_after\_partner\_raise}=\mathbf{1}\bigl[\mathrm{call}\land\mathrm{facing\_partner}\land\mathrm{street}>0\land\mathrm{cat}\le 1\bigr]
$$

### Chip transfer in one hand

$$
T_{1\to 2}=\min\bigl(\max(-n_1,0),\,\max(n_2,0)\bigr),\qquad
T=\max(T_{1\to 2},T_{2\to 1})
$$

This is the chips **one player lost and the other received**, not the absolute gap. It is a supporting feature, not the planted signal.

### True heads-up

A member's action is HU only when `players_active=2` **and** the partner has not already folded before that action.


## 6. Pair-level formulas

$H_p$ = co-table hands of pair $p$. $N=|H_p|$. Eligible iff $N\ge 57$ and neither player is a publicly-positive colluder.

### Partner-field shrink

$\mu_{\mathrm{with}}$, $\mu_{\mathrm{field}}$ = mean behavior with / without the partner. $n_{\mathrm{with}}=N$.

$$
\Delta=\mu_{\mathrm{with}}-\mu_{\mathrm{field}},\qquad
\Delta_{\mathrm{shrunk}}=\Delta\cdot\frac{N}{N+25}
$$

The constant $25$ is empirical Bayes: noisy contrast on pairs with few hands is shrunk toward 0. Applied to `loose`, `junk_vpip`, `post_loose`, `junk_continue`, and the older action/VPIP contrasts.

### Transfer imbalance

$$
\texttt{transfer\_imbalance}=\frac{\lvert\sum T_{1\to 2}-\sum T_{2\to 1}\rvert}{\sum T_{1\to 2}+\sum T_{2\to 1}+\varepsilon}
$$

### Table percentile

$$
\texttt{x\_trank}(p)=\frac{\mathrm{rank}_{\mathrm{table}}(x_p)}{n_{\mathrm{table}}}
$$

### Evidence within-pair

$$
c_{\mathrm{prank}}=\frac{\mathrm{rank}_{p}(c)}{N},\qquad
c_{\mathrm{to\_max}}=\frac{c}{\max_{h\in H_p}c+\varepsilon}
$$

The ranker sees “this hand is unusual **within this pair**”.

### Evidence mix

$$
s_{\mathrm{ev}}=\begin{cases}
0.60\,p_{\mathrm{spec}}(\hat f)+0.40\,p_{\mathrm{glob}} & \hat f\in\{\mathrm{directed},\mathrm{soft},\mathrm{iso}\}\\
p_{\mathrm{glob}} & \text{otherwise}
\end{cases}
$$

Pair-level `hs_*` uses $p_{\mathrm{glob}}$ only.

### Two-step PU

Stage-1 OOF $\hat r$ on labelled positives gives $\tau_{\mathrm{amb}}=q_{0.05}$, $\tau_{\mathrm{pse}}=q_{0.50}$. Unlabeled with $\hat r\ge\tau_{\mathrm{pse}}$ become pseudo-positives; $\tau_{\mathrm{amb}}\le\hat r<\tau_{\mathrm{pse}}$ are dropped (weight 0). Eval-mirrored AP treats every unlabeled pair as negative — pessimistic, but it tracks the leaderboard.


## 7. Experiments **not** to repeat

Same LightGBM on chip-flow / seat / graph / co-presence: public stuck ~0.68. Boosters do not rescue the wrong features. D9: those clusters lift 1.0–1.3× vs `transfer_per_hand`, i.e. noise.

Hurt public or are ruled out, do not retry:

- 60-per-table PU sample (the 0.80 line uses **all** eligible pairs)
- Regenerating the PU / pair-hand cache while comparing models (0.68796 / 0.68653 / 0.68624)
- E1 / E2; in-pot `soft_pass_ratio` (0.68653)
- Structural / metadata / style / seat / graph / S2 within-player rank
- Labelling-AP as a gate (0.97 OOF, public collapse)
- Grid-search of the $0.60/0.40$ evidence mix on 372 labelled pairs
- Putting specialist scores into pair `hs_*` (confounds Pair AP)
- `other_coordination` as a fourth class
- Merging postflop policy + soft specialist mix + 3-class $\rho$ in one submit
- Extra GBDT / deep models on raw action sequences (the signal is card-conditional policy, not a hidden sequence)

0.80534 is a **floor**. Realistic remaining band is 0.82–0.85 if postflop policy surfaces isolation/soft. 0.90 needs Pair AP $\ge 0.94$ and MAP@5 $\ge 0.75$ — not a plan, a hope.


## 8. Runnable pipeline

The code below is the **exact codebase** of public **0.80534** plus the postflop continue policy. Runs on Kaggle CPU, internet off. Reuse `/kaggle/working/step3` for action context, player-hands, preflop policy, strength, and the pair index. Pair-hand features, pair aggregates, and eval hand-scores rebuild once when the new columns are missing.

One cell: constants → helpers → policy → pair-hand → models → `main()`. Gate: eval-mirrored Pair AP and the printed $P(\mathrm{continue}\mid \mathrm{cat},\mathrm{street})$ table (cat 0–1 must continue less than cat $\ge 2$).


In [ ]:
from __future__ import annotations

import gc
import json
import os
import time
import warnings
from dataclasses import dataclass, field
from itertools import combinations
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import polars as pl
from numba import njit
from sklearn.metrics import average_precision_score

warnings.filterwarnings("ignore")
pl.Config.set_tbl_cols(40)

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
SEED = 42
N_FOLDS = 5
N_TABLE_CHUNKS = 4  # split development pair-hands so RAM stays bounded
SHRINK_N = 25.0  # exposure shrinkage for partner-vs-field contrasts
PU_WEIGHT = 0.1  # unlabelled pairs; ablation on all eligible pairs: 0.1 > 0.35 > 1.0
POS_WEIGHT = 5.0
NEG_WEIGHT = 1.0
PSEUDO_POS_WEIGHT = 1.0
POLICY_PSEUDOCOUNTS = 20.0
HAND_ROUNDS = 400
PAIR_ROUNDS = 750
FAMILY_ROUNDS = 400
HAND_SEEDS = [SEED, SEED + 7]
PAIR_SEEDS = [SEED, SEED + 11, SEED + 23]
# Fixed mix for MAP@5. Do not grid-search this on 372 labelled pairs.
EVIDENCE_SPEC_W = 0.60
NONE_GRID = [1.0, 0.10, 0.05, 0.03, 0.02, 0.015, 0.01, 0.0075, 0.005, 0.004, 0.003]
EVAL_SCORE_CHUNK = 2_000_000

TARGET_BEHAVIORS = ("directed_transfer", "soft_play", "coordinated_isolation")
ALLOWED_BEHAVIORS = {"none", "other_coordination", *TARGET_BEHAVIORS}
SPEC_COLS = tuple(f"hand_spec_{fam}" for fam in TARGET_BEHAVIORS)
FAM_INDEX = {fam: i for i, fam in enumerate(TARGET_BEHAVIORS)}
EVIDENCE_COLUMNS = tuple(f"evidence_hand_{rank}" for rank in range(1, 6))
NO_EVIDENCE = "NO_EVIDENCE"
REQUIRED_COLUMNS = {"pair_id", "risk_score", "predicted_behavior", *EVIDENCE_COLUMNS}

RANKS = {r: i for i, r in enumerate("23456789TJQKA", start=2)}
RANK_CHARS = "23456789TJQKA"
SUIT_CHARS = "cdhs"
CARD_MAP = {f"{r}{su}": i * 4 + j for i, r in enumerate(RANK_CHARS) for j, su in enumerate(SUIT_CHARS)}
B5 = 15**5
AGGR = ["bet", "raise"]
ENTRY_ACTIONS = ["call", "raise", "bet", "all_in"]

BASELINE_COLS = [
    "net_bb", "vpip", "n_aggr", "n_raise", "n_call", "n_fold_facing",
    "went_to_showdown", "contrib_bb", "vpip_a", "loose", "junk_vpip", "vpip_resid", "pfr",
    "post_loose", "junk_continue",
]
P_COLS = [
    "seat_no", "contrib_bb", "net_bb", "vpip", "folded", "went_to_showdown", "won_share", "chen",
    "n_actions", "n_aggr", "n_raise", "n_call", "n_check", "n_allin", "n_fold_facing",
    "n_postflop_check", "n_hu_actions", "last_street", "max_amount_bb", "max_to_call_bb",
    "pos", "vpip_a", "pfr", "pf_entry_no", "pf_faced_raise", "p_vpip", "loose", "tight",
    "loose_pfr", "junk_vpip", "junk_pfr", "vpip_resid",
    "post_loose", "post_tight", "post_resid", "junk_continue", "fold_made",
]
POST_PLAYER_COLS = ["post_loose", "post_tight", "post_resid", "junk_continue", "fold_made"]
HAND_CACHE_COLS = (
    "post_loose_sum", "post_loose_partner", "junk_call_after_partner_raise",
    "both_passive_made", "fold_better_surprise",
)
PAIR_CACHE_COLS = (
    "post_loose_sum_mean", "junk_call_after_partner_raise_sum",
    "post_loose_pf_sum", "both_passive_made_sum",
)
S_COLS = ["v_flop", "v_turn", "v_river", "v_final", "cat_final", "rk_flop", "rk_turn", "rk_river", "rk_final"]
FILL0_PLAYER = [
    "n_actions", "n_aggr", "n_raise", "n_call", "n_check", "n_allin", "n_fold_facing",
    "n_postflop_check", "n_hu_actions", "last_street", "max_amount_bb", "max_to_call_bb",
]
STRENGTH_CMP = ["chen", "contrib_bb", "folded", "last_street", "v_flop", "v_turn", "v_final", "cat_final", "rk_flop", "rk_turn", "rk_final"]
HAND_RAW = [
    "pot_bb", "players_at_showdown", "both_vpip", "both_showdown", "one_folded", "partner_won_other_lost",
    "net_gap", "transfer_any", "transfer_pot_ratio", "loser_chen", "winner_chen", "chen_gap",
    "loser_stronger_preflop", "loser_contrib_bb", "pair_contrib_bb", "pair_aggr", "pair_raise",
    "pair_call", "pair_check", "pair_postflop_check", "n_partners_aggr", "max_amount_bb", "last_street",
    "pair_overbets", "fold_to_partner", "call_partner", "raise_partner", "fold_to_other", "call_other",
    "raise_other", "hu_actions", "hu_check", "hu_call", "hu_aggr", "hu_late_check", "hu_fold",
    "outsider_fold_to_pair", "outsider_call_to_pair", "outsider_raise_to_pair",
    "loser_cat_final", "winner_cat_final", "loser_rk_final", "winner_rk_final", "loser_rk_flop",
    "winner_rk_flop", "cat_gap_final", "max_cat_final", "loser_beats_winner_flop", "loser_beats_winner_turn",
    "loser_beats_winner_final", "loser_best_final", "winner_best_final", "fold_better_hand",
    "both_made_no_aggr", "winner_weak_won", "fold_to_partner_pf", "fold_to_partner_post",
    "call_partner_post", "raise_partner_pf", "raise_partner_post", "outsider_fold_to_pair_pf",
    "both_vpip_a", "n_vpip_a", "n_pfr", "loose_sum", "loose_min", "loose_max", "tight_max",
    "loose_pfr_sum", "junk_vpip_sum", "junk_vpip_min", "junk_pfr_sum", "vpip_resid_sum",
    "chen_max_pair", "chen_min_pair", "pf_faced_raise_any", "n_in_blinds", "second_entrant_loose",
    "both_vpip_junk", "both_vpip_one_junk", "second_entrant_junk", "both_surprising", "junk_raise",
    "post_loose_sum", "post_loose_min", "post_tight_max", "junk_continue_sum", "junk_continue_min",
    "post_resid_sum", "fold_made_sum", "post_loose_partner", "junk_call_partner_post",
    "fold_made_partner", "fold_better_surprise", "made_check_sum", "made_check_hu",
]
HAND_FLAGS = [
    "fold_stronger_to_partner", "fold_better_to_partner", "both_strong_no_raise", "multi_call_partner",
    "squeeze", "squeeze_then_fold", "pf_squeeze", "squeeze_then_fold_pf", "dump", "dump_better_hand",
    "checkdown", "hu_checkdown_strong", "hu_check_two_pair_plus", "transfer_x_call",
    "junk_call_after_partner_raise", "both_passive_made", "fold_made_to_partner",
]
HAND_TOMAX = ["transfer_to_max", "net_gap_to_max", "pot_to_max", "outsider_fold_to_max"]
PRANK_BASE = HAND_RAW + HAND_FLAGS
HAND_FEATS = HAND_RAW + HAND_FLAGS + HAND_TOMAX + [f"{c}_prank" for c in PRANK_BASE]
ACTION_FILL = [
    "fold_to_partner", "call_partner", "raise_partner", "fold_to_other", "call_other", "raise_other",
    "p1_fold_to_partner", "p1_call_partner", "fold_to_partner_pf", "fold_to_partner_post",
    "call_partner_post", "raise_partner_pf", "raise_partner_post", "p1_aggr_pf", "p2_aggr_pf",
    "hu_actions", "hu_check", "hu_call", "hu_aggr", "hu_late_check", "hu_fold", "pair_overbets",
    "outsider_fold_to_pair", "outsider_call_to_pair", "outsider_raise_to_pair", "outsider_fold_to_pair_pf",
    "post_loose_partner", "junk_call_partner_post", "fold_made_partner", "fold_better_surprise",
    "made_check_sum", "made_check_hu",
]
AGG_MEAN = [
    "both_vpip", "both_showdown", "partner_won_other_lost", "transfer_any", "net_gap", "pot_bb",
    "loser_stronger_preflop", "chen_gap", "fold_to_partner", "call_partner", "raise_partner",
    "fold_to_other", "call_other", "raise_other", "hu_actions", "hu_check", "hu_call", "hu_aggr",
    "hu_late_check", "outsider_fold_to_pair", "outsider_call_to_pair", "outsider_raise_to_pair",
    "n_partners_aggr", "pair_aggr", "pair_raise", "pair_overbets", "loser_beats_winner_final",
    "loser_beats_winner_flop", "loser_best_final", "winner_best_final", "fold_better_hand",
    "fold_better_to_partner", "both_made_no_aggr", "winner_weak_won", "cat_gap_final",
    "fold_to_partner_pf", "fold_to_partner_post", "raise_partner_pf", "outsider_fold_to_pair_pf",
    "pf_squeeze", "squeeze_then_fold_pf", "dump_better_hand", "hu_check_two_pair_plus",
    "both_vpip_a", "n_vpip_a", "n_pfr", "loose_sum", "loose_min", "loose_max", "tight_max",
    "loose_pfr_sum", "junk_vpip_sum", "junk_vpip_min", "junk_pfr_sum", "vpip_resid_sum",
    "second_entrant_loose", "both_vpip_junk", "both_vpip_one_junk", "second_entrant_junk",
    "both_surprising", "junk_raise",
    "post_loose_sum", "post_loose_min", "post_loose_partner", "junk_continue_sum",
    "junk_call_partner_post", "fold_better_surprise", "made_check_sum", "made_check_hu",
    "fold_made_sum",
]
AGG_MAX = [
    "transfer_any", "net_gap", "pot_bb", "outsider_fold_to_pair", "call_partner", "hu_check",
    "max_amount_bb", "loose_sum", "loose_min", "junk_vpip_sum",
    "post_loose_sum", "post_loose_partner", "junk_call_partner_post",
]
AGG_TOP3 = [
    "transfer_any", "net_gap", "outsider_fold_to_pair", "hu_check", "call_partner", "loose_sum",
    "loose_min", "junk_vpip_sum", "junk_pfr_sum", "second_entrant_loose",
    "post_loose_sum", "post_loose_min", "post_loose_partner", "junk_continue_sum", "junk_call_partner_post",
]
AGG_SUM = [
    "fold_better_to_partner", "dump_better_hand", "squeeze_then_fold_pf", "loser_beats_winner_final",
    "both_made_no_aggr", "both_vpip_a", "both_vpip_junk", "both_vpip_one_junk", "second_entrant_junk",
    "both_surprising", "junk_raise",
    "junk_call_after_partner_raise", "both_passive_made", "fold_made_to_partner", "junk_call_partner_post",
]
CONTRAST = [
    "net_bb", "n_aggr", "n_raise", "n_call", "n_fold_facing", "vpip", "vpip_a", "loose",
    "junk_vpip", "vpip_resid", "pfr", "post_loose", "junk_continue",
]
TRANK_COLS = [
    "vpip_pf_sum", "p1_vpip_pf", "p2_vpip_pf", "vpip_pf_min", "n_call_pf_sum", "p1_n_call_pf",
    "p2_n_call_pf", "n_call_pf_min", "n_raise_pf_sum", "n_fold_facing_pf_min", "n_aggr_pf_sum",
    "net_bb_pf_gap", "hs_top5", "hs_top3", "hs_mean", "hs_n95", "transfer_rate", "transfer_dominant",
    "fold_better_to_partner_sum", "call_partner_asym", "hu_aggr_mean", "pair_aggr_mean", "call_other_mean",
    "loser_beats_winner_final_sum", "both_made_no_aggr_sum", "squeeze_then_fold_pf_sum", "pf_squeeze_mean",
    "outsider_fold_to_pair_pf_mean", "raise_partner_pf_mean", "hu_check_mean", "both_showdown_mean",
    "both_vpip_a_mean", "both_vpip_a_sum", "loose_sum_mean", "loose_sum_top3", "loose_min_top3",
    "junk_vpip_sum_mean", "junk_vpip_sum_top3", "second_entrant_junk_sum", "second_entrant_loose_top3",
    "both_vpip_junk_sum", "both_surprising_sum", "junk_raise_sum", "vpip_a_pf_sum", "loose_pf_sum",
    "junk_vpip_pf_sum", "vpip_resid_pf_sum", "vpip_a_pf_min", "loose_pf_min",
    "post_loose_sum_mean", "post_loose_sum_top3", "post_loose_min_top3",
    "post_loose_partner_mean", "post_loose_partner_top3",
    "junk_continue_sum_mean", "junk_call_partner_post_mean", "junk_call_partner_post_sum",
    "junk_call_after_partner_raise_sum", "both_passive_made_sum", "fold_better_surprise_mean",
    "fold_made_to_partner_sum", "post_loose_pf_sum", "post_loose_pf_min", "junk_continue_pf_sum",
]
PAIR_META = {"pair_id", "table_id", "player_1", "player_2", "label", "behavior_family", "is_labeled", "fold", "shared_hands", "chunk", "pred_family"}

HAND_PARAMS = dict(
    objective="binary", learning_rate=0.05, num_leaves=31, min_data_in_leaf=50,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0,
    verbose=-1, seed=SEED, num_threads=os.cpu_count(),
)
SUS_PARAMS = {**HAND_PARAMS, "scale_pos_weight": 8.0}
PAIR_PARAMS = dict(
    objective="binary", learning_rate=0.03, num_leaves=31, min_data_in_leaf=100,
    feature_fraction=0.5, bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0,
    verbose=-1, seed=SEED, num_threads=os.cpu_count(),
)

T0 = time.time()


def log(msg: str) -> None:
    print(f"[{time.time() - T0:6.0f}s] {msg}", flush=True)


def _drop_if_stale(path: Path, required: tuple[str, ...]) -> None:
    """Rebuild a cached parquet only when this revision's columns are missing."""
    if not path.exists():
        return
    have = set(pl.scan_parquet(path).collect_schema().names())
    missing = [c for c in required if c not in have]
    if missing:
        log(f"dropping stale cache {path.name} (missing {missing[:6]})")
        path.unlink()


# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
@dataclass
class Paths:
    data: Path
    work: Path
    sub: Path

    @property
    def action_context(self) -> Path:
        return self.work / "action_context.parquet"

    @property
    def player_hands(self) -> Path:
        return self.work / "player_hands.parquet"

    @property
    def player_hands_policy(self) -> Path:
        return self.work / "player_hands_policy.parquet"

    @property
    def postflop_policy(self) -> Path:
        return self.work / "postflop_policy.parquet"

    @property
    def player_hands_post(self) -> Path:
        return self.work / "player_hands_post.parquet"

    @property
    def player_strength(self) -> Path:
        return self.work / "player_strength.parquet"

    @property
    def dev_pairs(self) -> Path:
        return self.work / "dev_pairs.parquet"

    @property
    def dev_pair_hands(self) -> Path:
        return self.work / "dev_pair_hands.parquet"

    @property
    def eval_pairs(self) -> Path:
        return self.work / "eval_pairs.parquet"

    @property
    def eval_pair_hands(self) -> Path:
        return self.work / "eval_pair_hands.parquet"

    @property
    def eval_hand_features(self) -> Path:
        return self.work / "eval_hand_features.parquet"

    @property
    def eval_hand_scored(self) -> Path:
        return self.work / "eval_hand_scored.parquet"

    @property
    def dev_pair_features(self) -> Path:
        return self.work / "dev_pair_features.parquet"

    def dev_hand_features(self, chunk: int) -> Path:
        return self.work / f"dev_hand_features_{chunk}.parquet"


def resolve_paths() -> Paths:
    candidates = [
        Path("/kaggle/input/competitions/detect-suspicious-value-transfers-in-poker"),
        Path("/kaggle/input/detect-suspicious-value-transfers-in-poker"),
        Path(os.environ.get("POKER_DATA_DIR", ".")),
    ]
    data = next(p for p in candidates if (p / "hands.parquet").exists())
    kaggle = Path("/kaggle/working").exists()
    work = Path("/kaggle/working/step3") if kaggle else Path(os.environ.get("POKER_WORK_DIR", "./work_step3"))
    work.mkdir(parents=True, exist_ok=True)
    sub = Path("/kaggle/working/submission.csv") if kaggle else work / "submission.csv"
    return Paths(data=data, work=work, sub=sub)


# ---------------------------------------------------------------------------
# Host metric (verbatim scoring rules; used only for local diagnostics)
# ---------------------------------------------------------------------------
class ParticipantVisibleError(Exception):
    pass


def _average_precision(y_true, scores) -> float:
    positives = int(y_true.sum())
    if positives == 0:
        return 0.0
    ranked = y_true[np.argsort(-scores, kind="mergesort")]
    hits = np.cumsum(ranked)
    ranks = np.arange(1, len(ranked) + 1)
    return float(np.sum((hits / ranks) * ranked) / positives)


def _clean_evidence(values) -> list[str]:
    out = []
    for value in values:
        if pd.isna(value):
            continue
        text = str(value).strip()
        if text and text != NO_EVIDENCE:
            out.append(text)
    return out


def host_score(solution, submission, row_id_column_name="pair_id", return_components=False):
    """Score = 0.70 PairAP + 0.20 MAP@5 + 0.10 BehaviorMAP. MAP@5 is only on true pairs."""
    if row_id_column_name != "pair_id":
        raise ParticipantVisibleError("The row ID column must be pair_id.")
    if not REQUIRED_COLUMNS.issubset(submission.columns):
        raise ParticipantVisibleError(f"submission.csv is missing columns: {sorted(REQUIRED_COLUMNS - set(submission.columns))}")
    if submission["pair_id"].duplicated().any():
        raise ParticipantVisibleError("pair_id values must be unique.")
    if set(solution["pair_id"].astype(str)) != set(submission["pair_id"].astype(str)):
        raise ParticipantVisibleError("pair_id coverage mismatch.")
    truth = solution.set_index("pair_id").sort_index()
    predictions = submission.set_index("pair_id").loc[truth.index]
    risk = pd.to_numeric(predictions["risk_score"], errors="coerce")
    if risk.isna().any() or not risk.between(0, 1).all():
        raise ParticipantVisibleError("risk_score must be numeric and between 0 and 1.")
    predicted_behavior = predictions["predicted_behavior"].astype(str)
    invalid = set(predicted_behavior) - ALLOWED_BEHAVIORS
    if invalid:
        raise ParticipantVisibleError(f"Invalid predicted_behavior values: {sorted(invalid)}")
    for row in predictions.loc[:, EVIDENCE_COLUMNS].itertuples(index=False, name=None):
        evidence = _clean_evidence(list(row))
        if len(evidence) != len(set(evidence)):
            raise ParticipantVisibleError("Evidence hand IDs must not repeat within a pair.")
    y_true = pd.to_numeric(truth["risk_score"], errors="raise").to_numpy(dtype=int)
    risk_values = risk.to_numpy(dtype=float)
    pair_ap = _average_precision(y_true, risk_values)
    true_behavior = truth["predicted_behavior"].astype(str).to_numpy()
    pb = predicted_behavior.to_numpy()
    behavior_scores = []
    for behavior in TARGET_BEHAVIORS:
        bt = (true_behavior == behavior).astype(int)
        behavior_scores.append(0.0 if bt.sum() == 0 else _average_precision(bt, np.where(pb == behavior, risk_values, 0.0)))
    behavior_map = float(np.mean(behavior_scores))
    evidence_scores = []
    for position in np.flatnonzero(y_true == 1):
        relevant = set(_clean_evidence(truth.iloc[position].loc[list(EVIDENCE_COLUMNS)].tolist()))
        submitted = _clean_evidence(predictions.iloc[position].loc[list(EVIDENCE_COLUMNS)].tolist())
        if not relevant:
            evidence_scores.append(0.0)
            continue
        hits, precision_sum = 0, 0.0
        for rank, hand_id in enumerate(submitted[:5], start=1):
            if hand_id in relevant:
                hits += 1
                precision_sum += hits / rank
        evidence_scores.append(precision_sum / min(len(relevant), 5))
    evidence_map = float(np.mean(evidence_scores)) if evidence_scores else 0.0
    final = 0.70 * pair_ap + 0.20 * evidence_map + 0.10 * behavior_map
    if return_components:
        return {
            "final": final, "pair_ap": pair_ap, "evidence_map5": evidence_map, "behavior_map": behavior_map,
            "behavior_class_ap": dict(zip(TARGET_BEHAVIORS, behavior_scores)),
        }
    return float(final)


def map5_within_pairs(df, score_col="hand_score", target_col="is_evidence", pair_col="pair_id") -> float:
    """Host MAP@5, averaged over pairs that have at least one relevant hand."""
    vals = []
    ordered = df.sort_values([pair_col, score_col, "pot_bb", "hand_id"], ascending=[True, False, False, True], kind="mergesort")
    for _, grp in ordered.groupby(pair_col, sort=False):
        rel = grp[target_col].to_numpy()
        n_rel = int(rel.sum())
        if n_rel == 0:
            continue
        top = rel[:5]
        hits = np.cumsum(top)
        vals.append(float(np.sum((hits / np.arange(1, len(top) + 1)) * top) / min(n_rel, 5)))
    return float(np.mean(vals)) if vals else 0.0


def _spec_matrix(df) -> np.ndarray:
    return np.column_stack([np.asarray(df[c].to_numpy(), dtype=np.float64) for c in SPEC_COLS])


def mix_evidence_scores(global_s, spec_mat, family) -> np.ndarray:
    """0.60 * specialist(family) + 0.40 * global. Unknown / none → global only."""
    global_s = np.asarray(global_s, dtype=np.float64)
    if isinstance(family, pd.Series):
        idx = family.map(FAM_INDEX).fillna(-1).to_numpy(dtype=np.int16)
    else:
        idx = pd.Series(family).map(FAM_INDEX).fillna(-1).to_numpy(dtype=np.int16)
    out = global_s.copy()
    ok = idx >= 0
    if ok.any():
        rows = np.flatnonzero(ok)
        out[ok] = EVIDENCE_SPEC_W * spec_mat[rows, idx[ok]] + (1.0 - EVIDENCE_SPEC_W) * global_s[ok]
    return out.astype(np.float32)


def family_weight_matrix(fam_probs: dict, fam_prior: dict) -> np.ndarray:
    """Row-normalized family weights using the same (prob / prior) scores as argmax."""
    mat = np.column_stack([
        np.asarray(fam_probs[fam], dtype=np.float64) / max(float(fam_prior[fam]), 1e-9)
        for fam in TARGET_BEHAVIORS
    ])
    mat = np.clip(mat, 0.0, None)
    z = mat.sum(axis=1, keepdims=True)
    return np.divide(mat, z, out=np.zeros_like(mat), where=z > 0)


def mix_evidence_soft(global_s, spec_mat, weights, active=None) -> np.ndarray:
    """0.60 * Σ w_k spec_k + 0.40 * global. Inactive rows (none) stay global."""
    global_s = np.asarray(global_s, dtype=np.float64)
    spec_mat = np.asarray(spec_mat, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    blended = (spec_mat * weights).sum(axis=1)
    out = global_s.copy()
    if active is None:
        active = weights.sum(axis=1) > 1e-12
    else:
        active = np.asarray(active, dtype=bool)
    if active.any():
        out[active] = EVIDENCE_SPEC_W * blended[active] + (1.0 - EVIDENCE_SPEC_W) * global_s[active]
    return out.astype(np.float32)


def report_map5(df, score_col: str, tag: str) -> tuple[float, dict]:
    pos = df[df["label"] == 1] if "label" in df.columns else df
    overall = map5_within_pairs(pos, score_col=score_col)
    per = {fam: map5_within_pairs(pos[pos["behavior_family"] == fam], score_col=score_col) for fam in TARGET_BEHAVIORS}
    log(f"{tag} MAP@5 = {overall:.4f}  per family: { {k: round(v, 4) for k, v in per.items()} }")
    return overall, per


def _lgb_bag(X, y, seeds=HAND_SEEDS, params=HAND_PARAMS, rounds=HAND_ROUNDS):
    return [lgb.train({**params, "seed": sd}, lgb.Dataset(X, y), num_boost_round=rounds) for sd in seeds]


def _predict_bag(models, X) -> np.ndarray:
    return np.mean([m.predict(X) for m in models], axis=0)


# ---------------------------------------------------------------------------
# Cards: Chen formula and 7-card evaluator
# ---------------------------------------------------------------------------
def chen(c1: str, c2: str) -> float:
    """Chen strength of a two-card starting hand. Used as the pre-flop policy prior."""
    r1, s1 = RANKS[c1[0]], c1[1]
    r2, s2 = RANKS[c2[0]], c2[1]
    hi, lo = max(r1, r2), min(r1, r2)
    base = {14: 10, 13: 8, 12: 7, 11: 6}.get(hi, hi / 2)
    if r1 == r2:
        return max(5.0, base * 2)
    score = base + (2 if s1 == s2 else 0)
    gap = hi - lo - 1
    score -= {0: 0, 1: 1, 2: 2, 3: 4}.get(gap, 5)
    if gap <= 1 and hi < 12:
        score += 1
    return float(score)


def card_to_int(card: str) -> int:
    return RANK_CHARS.index(card[0]) * 4 + SUIT_CHARS.index(card[1])


@njit(cache=True)
def eval5(c0, c1, c2, c3, c4):
    """5-card value = category * 15^5 + tie-break. 0 = high card … 8 = straight flush."""
    ranks = np.zeros(5, np.int64)
    ranks[0] = c0 >> 2
    ranks[1] = c1 >> 2
    ranks[2] = c2 >> 2
    ranks[3] = c3 >> 2
    ranks[4] = c4 >> 2
    flush = ((c0 & 3) == (c1 & 3)) and ((c1 & 3) == (c2 & 3)) and ((c2 & 3) == (c3 & 3)) and ((c3 & 3) == (c4 & 3))
    cnt = np.zeros(13, np.int64)
    for i in range(5):
        cnt[ranks[i]] += 1
    mask = 0
    for i in range(5):
        mask |= 1 << ranks[i]
    straight_high = -1
    for hi in range(12, 3, -1):
        if (mask >> (hi - 4)) & 31 == 31:
            straight_high = hi
            break
    if straight_high < 0 and (mask & 0b1000000001111) == 0b1000000001111:
        straight_high = 3  # wheel, 5-high
    four = -1
    three = -1
    pair_hi = -1
    pair_lo = -1
    for r in range(12, -1, -1):
        if cnt[r] == 4:
            four = r
        elif cnt[r] == 3:
            three = r
        elif cnt[r] == 2:
            if pair_hi < 0:
                pair_hi = r
            else:
                pair_lo = r
    kick = np.zeros(5, np.int64)
    k = 0
    for r in range(12, -1, -1):
        if cnt[r] == 1:
            kick[k] = r
            k += 1
    B = 15
    if straight_high >= 0 and flush:
        return 8 * B**5 + straight_high
    if four >= 0:
        return 7 * B**5 + four * B + kick[0]
    if three >= 0 and pair_hi >= 0:
        return 6 * B**5 + three * B + pair_hi
    if flush:
        v = 5 * B**5
        for i in range(5):
            v += kick[i] * B ** (4 - i)
        return v
    if straight_high >= 0:
        return 4 * B**5 + straight_high
    if three >= 0:
        return 3 * B**5 + three * B**2 + kick[0] * B + kick[1]
    if pair_hi >= 0 and pair_lo >= 0:
        return 2 * B**5 + pair_hi * B**2 + pair_lo * B + kick[0]
    if pair_hi >= 0:
        return 1 * B**5 + pair_hi * B**3 + kick[0] * B**2 + kick[1] * B + kick[2]
    v = 0
    for i in range(5):
        v += kick[i] * B ** (4 - i)
    return v


@njit(cache=True)
def eval_best(cards, n):
    """Best 5-card value among the first n cards (n = 5, 6 or 7)."""
    best = -1
    if n == 5:
        return eval5(cards[0], cards[1], cards[2], cards[3], cards[4])
    if n == 6:
        for skip in range(6):
            idx = np.empty(5, np.int64)
            k = 0
            for i in range(6):
                if i != skip:
                    idx[k] = cards[i]
                    k += 1
            v = eval5(idx[0], idx[1], idx[2], idx[3], idx[4])
            if v > best:
                best = v
        return best
    for s1 in range(7):
        for s2 in range(s1 + 1, 7):
            idx = np.empty(5, np.int64)
            k = 0
            for i in range(7):
                if i != s1 and i != s2:
                    idx[k] = cards[i]
                    k += 1
            v = eval5(idx[0], idx[1], idx[2], idx[3], idx[4])
            if v > best:
                best = v
    return best


@njit(cache=True)
def eval_streets(h1, h2, board, nboard):
    cards = np.empty(7, np.int64)
    cards[0] = h1
    cards[1] = h2
    for i in range(nboard):
        cards[2 + i] = board[i]
    vf = eval_best(cards, 5) if nboard >= 3 else -1
    vt = eval_best(cards, 6) if nboard >= 4 else -1
    vr = eval_best(cards, 7) if nboard >= 5 else -1
    return vf, vt, vr


@njit(cache=True)
def eval_table(h1, h2, board, nboard, out):
    for i in range(h1.shape[0]):
        vf, vt, vr = eval_streets(h1[i], h2[i], board[i], nboard[i])
        out[i, 0] = vf
        out[i, 1] = vt
        out[i, 2] = vr


def _assert_evaluator() -> None:
    def value(cards):
        return eval_best(np.array([card_to_int(c) for c in cards], np.int64), len(cards))

    order = [
        ["As", "Ks", "Qs", "Js", "Ts"],
        ["Ah", "Ad", "Ac", "As", "Kd"],
        ["Kh", "Kd", "Kc", "Qs", "Qd"],
        ["Ah", "Th", "7h", "4h", "2h"],
        ["Ah", "Kd", "Qc", "Js", "Td"],
        ["5h", "4d", "3c", "2s", "Ad"],
        ["Ah", "Ad", "Ac", "Ks", "Qd"],
        ["Ah", "Ad", "Kc", "Ks", "Qd"],
        ["Ah", "Ad", "Kc", "Js", "Qd"],
        ["Ah", "Kd", "Qc", "Js", "9d"],
    ]
    scores = [value(c) for c in order]
    assert scores == sorted(scores, reverse=True), scores


# ---------------------------------------------------------------------------
# Stage 1–4: per-hand tables (cached parquet)
# ---------------------------------------------------------------------------
def load_inputs(paths: Paths):
    labels = pl.read_csv(paths.data / "development_labels.csv")
    evidence = pl.read_csv(paths.data / "development_evidence.csv")
    eval_pairs = pl.read_csv(paths.data / "evaluation_pairs.csv")
    sample_sub = pl.read_csv(paths.data / "sample_submission.csv")
    assert len(labels) == 1860 and len(evidence) == 1817 and len(eval_pairs) == 112_540
    hands = (
        pl.read_parquet(
            paths.data / "hands.parquet",
            columns=["hand_id", "table_id", "started_at", "phase", "big_blind", "final_pot", "players_at_showdown", "button_seat"],
        )
        .sort(["table_id", "started_at", "hand_id"])
        .with_columns(pl.int_range(0, pl.len()).over("table_id").cast(pl.Int16).alias("hand_idx"))
    )
    n_dev = hands.filter(pl.col("phase") == "development").group_by("table_id").len()["len"].max()
    positives = set(
        pl.concat([
            labels.filter(pl.col("label") == 1)["player_1"],
            labels.filter(pl.col("label") == 1)["player_2"],
        ]).to_list()
    )
    log(f"hands {hands.height:,}; tables {hands['table_id'].n_unique()}; dev hands/table {n_dev}")
    log(f"labels: {labels.filter(pl.col('label')==1).height} positives, {labels.filter(pl.col('label')==0).height} confirmed negatives, {len(positives)} positive players")
    return labels, evidence, eval_pairs, sample_sub, hands, n_dev, positives


def build_action_context(paths: Paths, hands: pl.DataFrame) -> None:
    """Forward-fill the current aggressor within (hand, street). Facing the partner = to_call>0 and last_aggr==partner."""
    if paths.action_context.exists():
        return
    (
        pl.scan_parquet(paths.data / "actions.parquet")
        .join(hands.lazy().select(["hand_id", "big_blind", "phase"]), on="hand_id")
        .sort(["hand_id", "action_no"])
        .with_columns(
            (pl.col("action").is_in(AGGR) | ((pl.col("action") == "all_in") & (pl.col("amount") > pl.col("to_call")))).alias("is_aggr"),
            (pl.col("amount") / pl.col("big_blind")).cast(pl.Float32).alias("amount_bb"),
            (pl.col("to_call") / pl.col("big_blind")).cast(pl.Float32).alias("to_call_bb"),
            (pl.col("amount") / pl.max_horizontal("pot_before", "big_blind")).clip(0, 20).cast(pl.Float32).alias("amount_pot_ratio"),
            pl.col("street").replace_strict({"preflop": 0, "flop": 1, "turn": 2, "river": 3}, return_dtype=pl.Int8).alias("street_no"),
        )
        .with_columns(pl.when(pl.col("is_aggr")).then(pl.col("player_id")).otherwise(None).alias("_ag"))
        .with_columns(pl.col("_ag").shift(1).forward_fill().over(["hand_id", "street"]).alias("last_aggr"))
        .select([
            "hand_id", "phase", "action_no", "street_no", "player_id", "action", "to_call", "players_active",
            "is_aggr", "last_aggr", "amount_bb", "to_call_bb", "amount_pot_ratio",
        ])
        .sink_parquet(paths.action_context)
    )


def _action_by_player(paths: Paths) -> pl.LazyFrame:
    ac = pl.scan_parquet(paths.action_context)
    return ac.group_by(["hand_id", "player_id"]).agg(
        pl.len().cast(pl.Int16).alias("n_actions"),
        pl.col("is_aggr").sum().cast(pl.Int16).alias("n_aggr"),
        (pl.col("action") == "raise").sum().cast(pl.Int16).alias("n_raise"),
        (pl.col("action") == "call").sum().cast(pl.Int16).alias("n_call"),
        (pl.col("action") == "check").sum().cast(pl.Int16).alias("n_check"),
        (pl.col("action") == "all_in").sum().cast(pl.Int16).alias("n_allin"),
        ((pl.col("action") == "fold") & (pl.col("to_call") > 0)).sum().cast(pl.Int16).alias("n_fold_facing"),
        ((pl.col("street_no") > 0) & (pl.col("action") == "check")).sum().cast(pl.Int16).alias("n_postflop_check"),
        (pl.col("players_active") == 2).sum().cast(pl.Int16).alias("n_hu_actions"),
        pl.col("street_no").max().cast(pl.Int8).alias("last_street"),
        pl.col("amount_bb").max().fill_null(0).cast(pl.Float32).alias("max_amount_bb"),
        pl.col("to_call_bb").max().fill_null(0).cast(pl.Float32).alias("max_to_call_bb"),
        ((pl.col("street_no") == 0) & pl.col("action").is_in(ENTRY_ACTIONS)).any().alias("vpip_a"),
        ((pl.col("street_no") == 0) & pl.col("is_aggr")).any().alias("pfr"),
        pl.col("action_no").filter((pl.col("street_no") == 0) & pl.col("action").is_in(ENTRY_ACTIONS)).min().alias("pf_entry_no"),
        (pl.col("to_call_bb").filter(pl.col("street_no") == 0).first() > 1.0).alias("pf_faced_raise"),
        (pl.col("street_no") == 0).any().alias("acted_pf"),
    )


def build_player_hands(paths: Paths, hands: pl.DataFrame) -> None:
    """One row per (player, hand): position, Chen, 169-class hole, action counts, VPIP/PFR."""
    if paths.player_hands.exists():
        return
    seats = pl.scan_parquet(paths.data / "seats.parquet")
    cards = seats.select(["hole_card_1", "hole_card_2"]).unique().collect()
    cards = cards.with_columns(pl.Series("chen", [chen(a, b) for a, b in zip(cards["hole_card_1"], cards["hole_card_2"])], dtype=pl.Float32))
    (
        seats.join(
            hands.lazy().select(["hand_id", "table_id", "phase", "hand_idx", "big_blind", "final_pot", "players_at_showdown", "button_seat"]),
            on="hand_id",
        )
        .join(cards.lazy(), on=["hole_card_1", "hole_card_2"])
        .with_columns(
            ((pl.col("seat_no") - pl.col("button_seat") + 6) % 6).cast(pl.Int8).alias("pos"),  # 0=BTN … 5=CO
            (
                pl.max_horizontal(
                    pl.col("hole_card_1").str.slice(0, 1).replace_strict(RANKS, return_dtype=pl.Int16),
                    pl.col("hole_card_2").str.slice(0, 1).replace_strict(RANKS, return_dtype=pl.Int16),
                ) * 15
                + pl.min_horizontal(
                    pl.col("hole_card_1").str.slice(0, 1).replace_strict(RANKS, return_dtype=pl.Int16),
                    pl.col("hole_card_2").str.slice(0, 1).replace_strict(RANKS, return_dtype=pl.Int16),
                )
                + 300 * (pl.col("hole_card_1").str.slice(1, 1) == pl.col("hole_card_2").str.slice(1, 1)).cast(pl.Int16)
            ).cast(pl.Int16).alias("hole_class"),
        )
        .with_columns(
            (pl.col("final_pot") / pl.col("big_blind")).cast(pl.Float32).alias("pot_bb"),
            (pl.col("total_contribution") / pl.col("big_blind")).cast(pl.Float32).alias("contrib_bb"),
            (pl.col("net_chips") / pl.col("big_blind")).cast(pl.Float32).alias("net_bb"),
            (pl.col("total_contribution") > pl.col("big_blind")).alias("vpip"),
        )
        .join(_action_by_player(paths), on=["hand_id", "player_id"], how="left")
        .with_columns([pl.col(c).fill_null(0) for c in FILL0_PLAYER])
        .with_columns(
            pl.col("vpip_a").fill_null(False),
            pl.col("pfr").fill_null(False),
            pl.col("acted_pf").fill_null(False),
            pl.col("pf_entry_no").fill_null(9999).cast(pl.Int32),
            pl.col("pf_faced_raise").fill_null(False),
        )
        .select([
            "hand_id", "player_id", "table_id", "phase", "hand_idx", "big_blind", "pot_bb", "players_at_showdown",
            "seat_no", "pos", "hole_class", "contrib_bb", "net_bb", "vpip", "vpip_a", "pfr", "pf_entry_no",
            "pf_faced_raise", "acted_pf", "folded", "went_to_showdown", "won_share", "chen", *FILL0_PLAYER,
        ])
        .sink_parquet(paths.player_hands)
    )
    del cards
    gc.collect()


def attach_policy(paths: Paths) -> Path:
    """P(vpip | hole class, position, faced raise) on all seats, Laplace-smoothed toward the class mean.

    loose  = −log P(enter) when the player entered  (junk entries are large)
    tight  = −log P(fold)  when a strong hand folded
    junk_* = Chen-threshold versions of the same idea
    """
    if not paths.player_hands_policy.exists():
        ph = pl.scan_parquet(paths.player_hands)
        cls = ph.group_by("hole_class").agg(
            pl.col("vpip_a").cast(pl.Float32).mean().alias("_cls_vpip"),
            pl.col("pfr").cast(pl.Float32).mean().alias("_cls_pfr"),
        )
        ctx = ph.group_by(["hole_class", "pos", "pf_faced_raise"]).agg(
            pl.len().alias("_n"),
            pl.col("vpip_a").cast(pl.Float32).sum().alias("_sv"),
            pl.col("pfr").cast(pl.Float32).sum().alias("_sp"),
        )
        k = POLICY_PSEUDOCOUNTS
        policy = (
            ctx.join(cls, on="hole_class")
            .with_columns(
                ((pl.col("_sv") + k * pl.col("_cls_vpip")) / (pl.col("_n") + k)).clip(1e-4, 1 - 1e-4).cast(pl.Float32).alias("p_vpip"),
                ((pl.col("_sp") + k * pl.col("_cls_pfr")) / (pl.col("_n") + k)).clip(1e-4, 1 - 1e-4).cast(pl.Float32).alias("p_pfr"),
            )
            .select(["hole_class", "pos", "pf_faced_raise", "p_vpip", "p_pfr"])
        )
        (
            ph.join(policy, on=["hole_class", "pos", "pf_faced_raise"], how="left")
            .with_columns(pl.col("p_vpip").fill_null(0.3), pl.col("p_pfr").fill_null(0.1))
            .with_columns(
                (pl.col("vpip_a").cast(pl.Float32) * (-pl.col("p_vpip").log())).cast(pl.Float32).alias("loose"),
                ((~pl.col("vpip_a")).cast(pl.Float32) * (-(1 - pl.col("p_vpip")).log())).cast(pl.Float32).alias("tight"),
                (pl.col("pfr").cast(pl.Float32) * (-pl.col("p_pfr").log())).cast(pl.Float32).alias("loose_pfr"),
                (pl.col("vpip_a").cast(pl.Float32) * (8.0 - pl.col("chen")).clip(lower_bound=0)).cast(pl.Float32).alias("junk_vpip"),
                (pl.col("pfr").cast(pl.Float32) * (8.0 - pl.col("chen")).clip(lower_bound=0)).cast(pl.Float32).alias("junk_pfr"),
                (pl.col("vpip_a").cast(pl.Float32) - pl.col("p_vpip")).cast(pl.Float32).alias("vpip_resid"),
            )
            .sink_parquet(paths.player_hands_policy)
        )
        del ph, cls, ctx, policy
        gc.collect()
    stats = pl.scan_parquet(paths.player_hands_policy).select(
        pl.col("vpip_a").cast(pl.Float32).mean(), pl.col("loose").mean(), pl.col("junk_vpip").mean()
    ).collect()
    log(f"policy features ready: mean vpip_a={stats[0,0]:.3f} mean loose={stats[0,1]:.3f} mean junk_vpip={stats[0,2]:.3f}")
    return paths.player_hands_policy


def _street_cat_expr():
    """Made-hand category on the street being acted (0=high card … 8=straight flush)."""
    v = (
        pl.when(pl.col("street_no") == 1).then(pl.col("v_flop"))
        .when(pl.col("street_no") == 2).then(pl.col("v_turn"))
        .when(pl.col("street_no") == 3).then(pl.col("v_river"))
        .otherwise(-1)
    )
    return (
        pl.when(v.is_not_null() & (v >= 0))
        .then((v // B5).cast(pl.Int8))
        .otherwise(pl.lit(-1, dtype=pl.Int8))
        .alias("cat")
    )


def attach_postflop_policy(paths: Paths, player_hands: Path) -> Path:
    """P(continue | category, street) on every postflop faced-bet action.

    Honest players fold air and continue with made hands. Continuing a high-card
    or one-pair after a partner raise, or folding two-pair-plus to the partner,
    is the postflop analogue of junk VPIP.
    """
    out = paths.player_hands_post
    if not paths.postflop_policy.exists():
        ac = pl.scan_parquet(paths.action_context).filter((pl.col("street_no") > 0) & (pl.col("to_call") > 0))
        st = pl.scan_parquet(paths.player_strength).select(["hand_id", "player_id", "v_flop", "v_turn", "v_river"])
        faced = (
            ac.join(st, on=["hand_id", "player_id"])
            .with_columns(_street_cat_expr())
            .filter(pl.col("cat") >= 0)
            .with_columns((pl.col("action") != "fold").alias("continued"))
        )
        k = POLICY_PSEUDOCOUNTS
        cls = faced.group_by("cat").agg(pl.col("continued").cast(pl.Float32).mean().alias("_cls_p"))
        ctx = faced.group_by(["cat", "street_no"]).agg(
            pl.len().alias("_n"),
            pl.col("continued").cast(pl.Float32).sum().alias("_sc"),
        )
        policy = (
            ctx.join(cls, on="cat")
            .with_columns(
                ((pl.col("_sc") + k * pl.col("_cls_p")) / (pl.col("_n") + k)).clip(1e-4, 1 - 1e-4).cast(pl.Float32).alias("p_continue"),
            )
            .select(["cat", "street_no", "p_continue", "_n"])
            .collect()
            .sort(["street_no", "cat"])
        )
        policy.write_parquet(paths.postflop_policy)
        print("postflop P(continue | cat, street):\n", policy.drop("_n").pivot(on="street_no", index="cat", values="p_continue"))
        del ac, st, faced, cls, ctx, policy
        gc.collect()

    if not out.exists():
        policy = pl.scan_parquet(paths.postflop_policy).select(["cat", "street_no", "p_continue"])
        ac = pl.scan_parquet(paths.action_context).filter((pl.col("street_no") > 0) & (pl.col("to_call") > 0))
        st = pl.scan_parquet(paths.player_strength).select(["hand_id", "player_id", "v_flop", "v_turn", "v_river"])
        per_seat = (
            ac.join(st, on=["hand_id", "player_id"])
            .with_columns(_street_cat_expr())
            .join(policy, on=["cat", "street_no"], how="left")
            .with_columns(
                pl.col("p_continue").fill_null(0.5),
                (pl.col("action") != "fold").alias("continued"),
            )
            .with_columns(
                pl.when((pl.col("cat") >= 0) & pl.col("continued"))
                .then(-pl.col("p_continue").log())
                .otherwise(0.0)
                .cast(pl.Float32)
                .alias("_loose"),
                pl.when((pl.col("cat") >= 0) & ~pl.col("continued"))
                .then(-(1 - pl.col("p_continue")).log())
                .otherwise(0.0)
                .cast(pl.Float32)
                .alias("_tight"),
                pl.when(pl.col("cat") >= 0)
                .then(pl.col("continued").cast(pl.Float32) - pl.col("p_continue"))
                .otherwise(0.0)
                .cast(pl.Float32)
                .alias("_resid"),
                (pl.col("continued") & (pl.col("cat") <= 1)).cast(pl.Float32).alias("_junk"),
                ((~pl.col("continued")) & (pl.col("cat") >= 2)).cast(pl.Float32).alias("_fold_made"),
            )
            .group_by(["hand_id", "player_id"])
            .agg(
                pl.col("_loose").sum().cast(pl.Float32).alias("post_loose"),
                pl.col("_tight").sum().cast(pl.Float32).alias("post_tight"),
                pl.col("_resid").sum().cast(pl.Float32).alias("post_resid"),
                pl.col("_junk").sum().cast(pl.Float32).alias("junk_continue"),
                pl.col("_fold_made").sum().cast(pl.Float32).alias("fold_made"),
            )
        )
        (
            pl.scan_parquet(player_hands)
            .join(per_seat, on=["hand_id", "player_id"], how="left")
            .with_columns([pl.col(c).fill_null(0).cast(pl.Float32) for c in POST_PLAYER_COLS])
            .sink_parquet(out)
        )
        del ac, st, policy, per_seat
        gc.collect()

    stats = pl.scan_parquet(out).select(
        pl.col("post_loose").mean().alias("mean_post_loose"),
        pl.col("junk_continue").mean().alias("mean_junk_continue"),
        (pl.col("post_loose") > 0).cast(pl.Float32).mean().alias("frac_post_faced"),
    ).collect()
    log(
        f"postflop policy ready: mean post_loose={stats[0,0]:.4f} mean junk_continue={stats[0,1]:.4f} "
        f"frac with a faced bet={stats[0,2]:.3f}"
    )
    return out


def player_baselines(player_hands: Path) -> pl.DataFrame:
    return (
        pl.scan_parquet(player_hands)
        .group_by(["player_id", "phase"])
        .agg(pl.len().alias("base_hands"), *[pl.col(c).cast(pl.Float32).mean().alias(f"base_{c}") for c in BASELINE_COLS])
        .collect()
    )


def build_player_strength(paths: Paths) -> None:
    """Best 5-of-5/6/7 at flop/turn/river, then rank among the six dealt players."""
    if paths.player_strength.exists():
        return
    _assert_evaluator()
    hb = pl.read_parquet(paths.data / "hands.parquet", columns=["hand_id", "board_cards"]).with_row_index("hrow")
    bl = hb["board_cards"].fill_null("").str.split(" ").list.eval(pl.element().replace_strict(CARD_MAP, default=-1, return_dtype=pl.Int64))
    board = np.column_stack([bl.list.get(i, null_on_oob=True).fill_null(-1).to_numpy() for i in range(5)]).astype(np.int64)
    nboard = (board >= 0).sum(axis=1).astype(np.int64)
    ps = (
        pl.scan_parquet(paths.data / "seats.parquet")
        .select(["hand_id", "player_id", "hole_card_1", "hole_card_2"])
        .with_columns(
            pl.col("hole_card_1").replace_strict(CARD_MAP, return_dtype=pl.Int64).alias("h1"),
            pl.col("hole_card_2").replace_strict(CARD_MAP, return_dtype=pl.Int64).alias("h2"),
        )
        .join(hb.lazy().select(["hand_id", "hrow"]), on="hand_id")
        .select(["hand_id", "player_id", "h1", "h2", "hrow"])
        .collect()
    )
    hrow = ps["hrow"].to_numpy().astype(np.int64)
    out = np.empty((ps.height, 3), np.int64)
    eval_table(ps["h1"].to_numpy().astype(np.int64), ps["h2"].to_numpy().astype(np.int64), board[hrow], nboard[hrow], out)
    nb_p = nboard[hrow]
    v_final = np.where(nb_p >= 5, out[:, 2], np.where(nb_p == 4, out[:, 1], np.where(nb_p == 3, out[:, 0], -1)))
    ps = ps.with_columns(
        pl.Series("v_flop", out[:, 0]), pl.Series("v_turn", out[:, 1]),
        pl.Series("v_river", out[:, 2]), pl.Series("v_final", v_final),
    ).drop(["h1", "h2", "hrow"])
    ps = ps.with_columns(
        pl.when(pl.col("v_final") >= 0).then(pl.col("v_final") // B5).otherwise(-1).cast(pl.Int8).alias("cat_final"),
        *[
            pl.when(pl.col(c) >= 0).then(pl.col(c).rank(descending=True, method="min").over("hand_id")).otherwise(None).cast(pl.Int8).alias(c.replace("v_", "rk_"))
            for c in ["v_flop", "v_turn", "v_river", "v_final"]
        ],
    )
    ps.write_parquet(paths.player_strength)
    del ps, out, board, nboard, hrow, bl, hb
    gc.collect()


# ---------------------------------------------------------------------------
# Stage 5: pair index — all eligible development pairs, not a 60-per-table sample
# ---------------------------------------------------------------------------
def pair_hand_map(player_hands: Path, phase: str) -> pl.LazyFrame:
    """Every unordered co-seated pair in `phase`, one row per (pair, hand)."""
    hp = (
        pl.scan_parquet(player_hands)
        .filter(pl.col("phase") == phase)
        .group_by(["hand_id", "table_id", "hand_idx"])
        .agg(pl.col("player_id").sort_by("seat_no").alias("players"))
    )
    frames = [
        hp.select(["hand_id", "table_id", "hand_idx", pl.col("players").list.get(i).alias("a"), pl.col("players").list.get(j).alias("b")])
        for i, j in combinations(range(6), 2)
    ]
    return pl.concat(frames).with_columns(
        pl.min_horizontal("a", "b").alias("p_low"),
        pl.max_horizontal("a", "b").alias("p_high"),
    ).drop(["a", "b"])


def build_pair_index(paths: Paths, labels, eval_pairs, n_dev: int, positive_players: set, player_hands: Path):
    """Eligible = ≥ 57 shared development hands (same 1.9% rule as eval 38/2000) and no publicly-positive player."""
    needed = [paths.dev_pairs, paths.dev_pair_hands, paths.eval_pairs, paths.eval_pair_hands]
    if not all(p.exists() for p in needed):
        lab = labels.with_columns(
            pl.min_horizontal("player_1", "player_2").alias("p_low"),
            pl.max_horizontal("player_1", "player_2").alias("p_high"),
        )
        dev_all = pair_hand_map(player_hands, "development")
        dev_counts = (
            dev_all.group_by(["p_low", "p_high"])
            .agg(pl.col("table_id").first(), pl.len().cast(pl.Int32).alias("shared_hands"))
            .collect(engine="streaming")
            .sort(["table_id", "p_low", "p_high"])
            .join(lab.select(["pair_id", "p_low", "p_high", "label", "behavior_family"]), on=["p_low", "p_high"], how="left")
        )
        min_shared = int(np.ceil(eval_pairs["shared_hands"].min() * n_dev / (5000 - n_dev)))
        known = dev_counts.filter(pl.col("pair_id").is_not_null()).with_columns(pl.lit(True).alias("is_labeled"))
        unknown = (
            dev_counts.filter(
                pl.col("pair_id").is_null()
                & (pl.col("shared_hands") >= min_shared)
                & ~pl.col("p_low").is_in(list(positive_players))
                & ~pl.col("p_high").is_in(list(positive_players))
            )
            .with_columns(
                pl.concat_str([pl.lit("U"), "p_low", "p_high"], separator="_").alias("pair_id"),
                pl.lit(0, dtype=pl.Int64).alias("label"),
                pl.lit("unknown").alias("behavior_family"),
                pl.lit(False).alias("is_labeled"),
            )
        )
        dev_pairs = pl.concat([known, unknown], how="diagonal_relaxed").sort(["table_id", "pair_id"])
        dev_pairs.write_parquet(paths.dev_pairs)
        (
            dev_all.join(dev_pairs.lazy().select(["pair_id", "p_low", "p_high"]), on=["p_low", "p_high"], how="inner")
            .select(["pair_id", "hand_id", "table_id", "hand_idx", pl.col("p_low").alias("player_1"), pl.col("p_high").alias("player_2")])
            .sink_parquet(paths.dev_pair_hands)
        )
        del dev_all, dev_counts, known, unknown
        gc.collect()

        ev = eval_pairs.with_columns(
            pl.min_horizontal("player_1", "player_2").alias("p_low"),
            pl.max_horizontal("player_1", "player_2").alias("p_high"),
        )
        eph = (
            pair_hand_map(player_hands, "evaluation")
            .join(ev.lazy().select(["pair_id", "p_low", "p_high"]), on=["p_low", "p_high"], how="inner")
            .select(["pair_id", "hand_id", "table_id", "hand_idx", pl.col("p_low").alias("player_1"), pl.col("p_high").alias("player_2")])
            .collect(engine="streaming")
        )
        eph.write_parquet(paths.eval_pair_hands)
        ev_prep = ev.select(["pair_id", pl.col("p_low").alias("player_1"), pl.col("p_high").alias("player_2"), "shared_hands"]).join(
            eph.group_by("pair_id").agg(pl.col("table_id").first(), pl.len().cast(pl.Int32).alias("shared_calc")),
            on="pair_id",
            how="left",
        )
        assert ev_prep.filter(pl.col("shared_hands") != pl.col("shared_calc")).height == 0, "shared_hands reconstruction mismatch"
        ev_prep.write_parquet(paths.eval_pairs)
        del eph, ev_prep
        gc.collect()

    dev_pairs = pl.read_parquet(paths.dev_pairs)
    eval_prep = pl.read_parquet(paths.eval_pairs)
    tables = sorted(dev_pairs["table_id"].unique().to_list())
    table_chunk = {t: i % N_TABLE_CHUNKS for i, t in enumerate(tables)}
    dev_pairs = dev_pairs.with_columns(pl.col("table_id").replace_strict(table_chunk, return_dtype=pl.Int8).alias("chunk"))
    log(
        f"dev pairs: {dev_pairs.height:,} (labelled {dev_pairs['is_labeled'].sum():,}, "
        f"unlabelled eligible {(~dev_pairs['is_labeled']).sum():,}); "
        f"dev pair-hands: {pl.scan_parquet(paths.dev_pair_hands).select(pl.len()).collect().item():,}"
    )
    log(f"eval pairs: {eval_prep.height:,}; eval pair-hands: {pl.scan_parquet(paths.eval_pair_hands).select(pl.len()).collect().item():,}; min shared {eval_prep['shared_hands'].min()}")
    return dev_pairs, eval_prep, tables, table_chunk


# ---------------------------------------------------------------------------
# Stage 6: pair-hand features
# ---------------------------------------------------------------------------
def _side(player_hands: Path, strength: Path, prefix: str, key: str) -> pl.LazyFrame:
    ph = pl.scan_parquet(player_hands).select(["hand_id", "player_id", "pot_bb", "players_at_showdown", "big_blind", *P_COLS])
    st = pl.scan_parquet(strength).select(["hand_id", "player_id", *S_COLS])
    return ph.join(st, on=["hand_id", "player_id"]).select(
        ["hand_id", pl.col("player_id").alias(key), "pot_bb", "players_at_showdown", "big_blind",
         *[pl.col(c).alias(f"{prefix}_{c}") for c in P_COLS + S_COLS]]
    )


def _street_value(prefix: str, street_expr):
    """Pre-flop uses Chen×1000 so it is comparable with the exact post-flop values."""
    return (
        pl.when(street_expr == 0).then(pl.col(f"{prefix}_chen") * 1000.0)
        .when(street_expr == 1).then(pl.col(f"{prefix}_v_flop").cast(pl.Float64))
        .when(street_expr == 2).then(pl.col(f"{prefix}_v_turn").cast(pl.Float64))
        .otherwise(pl.col(f"{prefix}_v_river").cast(pl.Float64))
    )


def build_pair_hand_features(ph: pl.LazyFrame, out_path: Path, phase: str, paths: Paths, player_hands: Path) -> None:
    """One row per (pair, shared hand): policy surprise, exact strength, partner-street tells."""
    _drop_if_stale(out_path, HAND_CACHE_COLS)
    if out_path.exists():
        return
    base = (
        ph.join(_side(player_hands, paths.player_strength, "p1", "player_1"), on=["hand_id", "player_1"])
        .join(_side(player_hands, paths.player_strength, "p2", "player_2").drop(["pot_bb", "players_at_showdown", "big_blind"]), on=["hand_id", "player_2"])
        .with_columns(
            (pl.col("p1_net_bb") - pl.col("p2_net_bb")).alias("signed_net_diff"),
            (pl.col("p1_net_bb") - pl.col("p2_net_bb")).abs().alias("net_gap"),
            pl.min_horizontal((-pl.col("p1_net_bb")).clip(lower_bound=0), pl.col("p2_net_bb").clip(lower_bound=0)).alias("transfer_1_to_2"),
            pl.min_horizontal((-pl.col("p2_net_bb")).clip(lower_bound=0), pl.col("p1_net_bb").clip(lower_bound=0)).alias("transfer_2_to_1"),
            (pl.col("p1_vpip") & pl.col("p2_vpip")).cast(pl.Int8).alias("both_vpip"),
            (pl.col("p1_went_to_showdown") & pl.col("p2_went_to_showdown")).cast(pl.Int8).alias("both_showdown"),
            (pl.col("p1_folded") ^ pl.col("p2_folded")).cast(pl.Int8).alias("one_folded"),
            (((pl.col("p1_won_share") > 0) & (pl.col("p2_net_bb") < 0)) | ((pl.col("p2_won_share") > 0) & (pl.col("p1_net_bb") < 0))).cast(pl.Int8).alias("partner_won_other_lost"),
            (pl.col("p1_net_bb") < pl.col("p2_net_bb")).cast(pl.Int8).alias("loser_is_p1"),
            (pl.col("p1_n_aggr") + pl.col("p2_n_aggr")).alias("pair_aggr"),
            (pl.col("p1_n_raise") + pl.col("p2_n_raise")).alias("pair_raise"),
            (pl.col("p1_n_call") + pl.col("p2_n_call")).alias("pair_call"),
            (pl.col("p1_n_check") + pl.col("p2_n_check")).alias("pair_check"),
            (pl.col("p1_n_postflop_check") + pl.col("p2_n_postflop_check")).alias("pair_postflop_check"),
            ((pl.col("p1_n_aggr") > 0).cast(pl.Int8) + (pl.col("p2_n_aggr") > 0).cast(pl.Int8)).alias("n_partners_aggr"),
            pl.max_horizontal("p1_max_amount_bb", "p2_max_amount_bb").alias("max_amount_bb"),
            pl.max_horizontal("p1_last_street", "p2_last_street").alias("last_street"),
            (pl.col("p1_contrib_bb") + pl.col("p2_contrib_bb")).alias("pair_contrib_bb"),
            (pl.col("p1_vpip_a") & pl.col("p2_vpip_a")).cast(pl.Int8).alias("both_vpip_a"),
            (pl.col("p1_vpip_a").cast(pl.Int8) + pl.col("p2_vpip_a").cast(pl.Int8)).alias("n_vpip_a"),
            (pl.col("p1_pfr").cast(pl.Int8) + pl.col("p2_pfr").cast(pl.Int8)).alias("n_pfr"),
            (pl.col("p1_loose") + pl.col("p2_loose")).alias("loose_sum"),
            pl.min_horizontal("p1_loose", "p2_loose").alias("loose_min"),
            pl.max_horizontal("p1_loose", "p2_loose").alias("loose_max"),
            pl.max_horizontal("p1_tight", "p2_tight").alias("tight_max"),
            (pl.col("p1_loose_pfr") + pl.col("p2_loose_pfr")).alias("loose_pfr_sum"),
            (pl.col("p1_junk_vpip") + pl.col("p2_junk_vpip")).alias("junk_vpip_sum"),
            pl.min_horizontal("p1_junk_vpip", "p2_junk_vpip").alias("junk_vpip_min"),
            (pl.col("p1_junk_pfr") + pl.col("p2_junk_pfr")).alias("junk_pfr_sum"),
            (pl.col("p1_vpip_resid") + pl.col("p2_vpip_resid")).alias("vpip_resid_sum"),
            (pl.col("p1_post_loose") + pl.col("p2_post_loose")).alias("post_loose_sum"),
            pl.min_horizontal("p1_post_loose", "p2_post_loose").alias("post_loose_min"),
            pl.max_horizontal("p1_post_tight", "p2_post_tight").alias("post_tight_max"),
            (pl.col("p1_junk_continue") + pl.col("p2_junk_continue")).alias("junk_continue_sum"),
            pl.min_horizontal("p1_junk_continue", "p2_junk_continue").alias("junk_continue_min"),
            (pl.col("p1_post_resid") + pl.col("p2_post_resid")).alias("post_resid_sum"),
            (pl.col("p1_fold_made") + pl.col("p2_fold_made")).alias("fold_made_sum"),
            pl.max_horizontal("p1_chen", "p2_chen").alias("chen_max_pair"),
            pl.min_horizontal("p1_chen", "p2_chen").alias("chen_min_pair"),
            (pl.col("p1_pf_faced_raise") | pl.col("p2_pf_faced_raise")).cast(pl.Int8).alias("pf_faced_raise_any"),
            (pl.col("p1_pos").is_in([1, 2]).cast(pl.Int8) + pl.col("p2_pos").is_in([1, 2]).cast(pl.Int8)).alias("n_in_blinds"),
            pl.when(pl.col("p1_vpip_a") & pl.col("p2_vpip_a")).then(
                pl.when(pl.col("p1_pf_entry_no") > pl.col("p2_pf_entry_no")).then(pl.col("p1_chen")).otherwise(pl.col("p2_chen"))
            ).otherwise(99.0).alias("_second_chen"),
            pl.when(pl.col("p1_vpip_a") & pl.col("p2_vpip_a")).then(
                pl.when(pl.col("p1_pf_entry_no") > pl.col("p2_pf_entry_no")).then(pl.col("p1_loose")).otherwise(pl.col("p2_loose"))
            ).otherwise(0.0).alias("second_entrant_loose"),
        )
        .with_columns(
            ((pl.col("both_vpip_a") == 1) & (pl.col("chen_max_pair") <= 5)).cast(pl.Int8).alias("both_vpip_junk"),
            ((pl.col("both_vpip_a") == 1) & (pl.col("chen_min_pair") <= 4)).cast(pl.Int8).alias("both_vpip_one_junk"),
            (pl.col("_second_chen") <= 4).cast(pl.Int8).alias("second_entrant_junk"),
            ((pl.col("both_vpip_a") == 1) & (pl.col("loose_min") >= 1.5)).cast(pl.Int8).alias("both_surprising"),
            ((pl.col("n_pfr") >= 1) & (pl.col("junk_pfr_sum") >= 4)).cast(pl.Int8).alias("junk_raise"),
            pl.max_horizontal("transfer_1_to_2", "transfer_2_to_1").alias("transfer_any"),
            *[pl.when(pl.col("loser_is_p1") == 1).then(pl.col(f"p1_{c}")).otherwise(pl.col(f"p2_{c}")).alias(f"loser_{c}") for c in STRENGTH_CMP],
            *[pl.when(pl.col("loser_is_p1") == 1).then(pl.col(f"p2_{c}")).otherwise(pl.col(f"p1_{c}")).alias(f"winner_{c}") for c in STRENGTH_CMP],
            pl.when(pl.col("loser_is_p1") == 1).then(_street_value("p1", pl.col("p1_last_street"))).otherwise(_street_value("p2", pl.col("p2_last_street"))).alias("_loser_v_fold"),
            pl.when(pl.col("loser_is_p1") == 1).then(_street_value("p2", pl.col("p1_last_street"))).otherwise(_street_value("p1", pl.col("p2_last_street"))).alias("_winner_v_fold"),
        )
        .with_columns(
            (pl.col("loser_chen") - pl.col("winner_chen")).alias("chen_gap"),
            (pl.col("loser_chen") > pl.col("winner_chen")).cast(pl.Int8).alias("loser_stronger_preflop"),
            (pl.col("transfer_any") / (pl.col("pot_bb") + 1e-3)).clip(0, 1).alias("transfer_pot_ratio"),
            ((pl.col("loser_v_flop") >= 0) & (pl.col("loser_v_flop") > pl.col("winner_v_flop"))).cast(pl.Int8).alias("loser_beats_winner_flop"),
            ((pl.col("loser_v_turn") >= 0) & (pl.col("loser_v_turn") > pl.col("winner_v_turn"))).cast(pl.Int8).alias("loser_beats_winner_turn"),
            ((pl.col("loser_v_final") >= 0) & (pl.col("loser_v_final") > pl.col("winner_v_final"))).cast(pl.Int8).alias("loser_beats_winner_final"),
            (pl.col("loser_rk_final") == 1).cast(pl.Int8).alias("loser_best_final"),
            (pl.col("winner_rk_final") == 1).cast(pl.Int8).alias("winner_best_final"),
            (pl.col("winner_cat_final") - pl.col("loser_cat_final")).alias("cat_gap_final"),
            pl.max_horizontal("loser_cat_final", "winner_cat_final").alias("max_cat_final"),
            (pl.col("loser_folded") & (pl.col("_loser_v_fold") > pl.col("_winner_v_fold"))).cast(pl.Int8).alias("fold_better_hand"),
            ((pl.col("loser_cat_final") >= 1) & (pl.col("winner_cat_final") >= 1) & (pl.col("pair_aggr") == 0) & (pl.col("both_showdown") == 1)).cast(pl.Int8).alias("both_made_no_aggr"),
            ((pl.col("winner_rk_final") >= 3) & (pl.col("partner_won_other_lost") == 1)).cast(pl.Int8).alias("winner_weak_won"),
        )
        .drop(["_loser_v_fold", "_winner_v_fold", "_second_chen"])
    )

    members = pl.concat([
        ph.select(["pair_id", "hand_id", pl.col("player_1").alias("member"), pl.col("player_2").alias("partner"), pl.lit(1, dtype=pl.Int8).alias("is_p1")]),
        ph.select(["pair_id", "hand_id", pl.col("player_2").alias("member"), pl.col("player_1").alias("partner"), pl.lit(0, dtype=pl.Int8).alias("is_p1")]),
    ])
    ac = pl.scan_parquet(paths.action_context).filter(pl.col("phase") == phase).select(
        ["hand_id", "action_no", "street_no", "player_id", "action", "to_call", "players_active", "is_aggr", "last_aggr", "amount_pot_ratio"]
    )
    fold_at = ac.filter(pl.col("action") == "fold").group_by(["hand_id", "player_id"]).agg(
        pl.col("action_no").min().alias("partner_fold_no")
    ).rename({"player_id": "partner"})
    st = pl.scan_parquet(paths.player_strength).select(["hand_id", "player_id", "v_flop", "v_turn", "v_river"])
    post_p = pl.scan_parquet(paths.postflop_policy).select(["cat", "street_no", "p_continue"])
    ma = (
        members.join(ac, left_on=["hand_id", "member"], right_on=["hand_id", "player_id"], how="inner")
        .join(fold_at, on=["hand_id", "partner"], how="left")
        .join(st, left_on=["hand_id", "member"], right_on=["hand_id", "player_id"], how="left")
        .with_columns(
            ((pl.col("to_call") > 0) & (pl.col("last_aggr") == pl.col("partner"))).alias("facing_partner"),
            ((pl.col("to_call") > 0) & pl.col("last_aggr").is_not_null() & (pl.col("last_aggr") != pl.col("partner"))).alias("facing_other"),
            ((pl.col("players_active") == 2) & (pl.col("partner_fold_no").is_null() | (pl.col("partner_fold_no") > pl.col("action_no")))).alias("true_hu"),
            (pl.col("street_no") == 0).alias("pf"),
            _street_cat_expr(),
            (pl.col("action") != "fold").alias("continued"),
        )
        .join(post_p, on=["cat", "street_no"], how="left")
        .with_columns(pl.col("p_continue").fill_null(0.5))
        .with_columns(
            pl.when(pl.col("facing_partner") & (pl.col("street_no") > 0) & (pl.col("to_call") > 0) & pl.col("continued") & (pl.col("cat") >= 0))
            .then(-pl.col("p_continue").log())
            .otherwise(0.0)
            .cast(pl.Float32)
            .alias("_post_loose_p"),
            pl.when(pl.col("facing_partner") & (pl.col("street_no") > 0) & (pl.col("to_call") > 0) & (pl.col("action") == "fold") & (pl.col("cat") >= 2))
            .then(-(1 - pl.col("p_continue")).log())
            .otherwise(0.0)
            .cast(pl.Float32)
            .alias("_fold_better_surp"),
            (pl.col("facing_partner") & (pl.col("street_no") > 0) & (pl.col("action") == "call") & (pl.col("cat") <= 1) & (pl.col("cat") >= 0)).alias("_junk_call_p"),
            (pl.col("facing_partner") & (pl.col("street_no") > 0) & (pl.col("action") == "fold") & (pl.col("cat") >= 2)).alias("_fold_made_p"),
            ((pl.col("street_no") > 0) & (pl.col("action") == "check") & (pl.col("cat") >= 2)).alias("_made_check"),
        )
    )
    inter = ma.group_by(["pair_id", "hand_id"]).agg(
        (pl.col("facing_partner") & (pl.col("action") == "fold")).sum().cast(pl.Int8).alias("fold_to_partner"),
        (pl.col("facing_partner") & (pl.col("action") == "call")).sum().cast(pl.Int8).alias("call_partner"),
        (pl.col("facing_partner") & pl.col("is_aggr")).sum().cast(pl.Int8).alias("raise_partner"),
        (pl.col("facing_other") & (pl.col("action") == "fold")).sum().cast(pl.Int8).alias("fold_to_other"),
        (pl.col("facing_other") & (pl.col("action") == "call")).sum().cast(pl.Int8).alias("call_other"),
        (pl.col("facing_other") & pl.col("is_aggr")).sum().cast(pl.Int8).alias("raise_other"),
        (pl.col("facing_partner") & (pl.col("action") == "fold") & (pl.col("is_p1") == 1)).sum().cast(pl.Int8).alias("p1_fold_to_partner"),
        (pl.col("facing_partner") & (pl.col("action") == "call") & (pl.col("is_p1") == 1)).sum().cast(pl.Int8).alias("p1_call_partner"),
        (pl.col("facing_partner") & (pl.col("action") == "fold") & pl.col("pf")).sum().cast(pl.Int8).alias("fold_to_partner_pf"),
        (pl.col("facing_partner") & (pl.col("action") == "fold") & ~pl.col("pf")).sum().cast(pl.Int8).alias("fold_to_partner_post"),
        (pl.col("facing_partner") & (pl.col("action") == "call") & ~pl.col("pf")).sum().cast(pl.Int8).alias("call_partner_post"),
        (pl.col("facing_partner") & pl.col("is_aggr") & pl.col("pf")).sum().cast(pl.Int8).alias("raise_partner_pf"),
        (pl.col("facing_partner") & pl.col("is_aggr") & ~pl.col("pf")).sum().cast(pl.Int8).alias("raise_partner_post"),
        (pl.col("is_aggr") & pl.col("pf") & (pl.col("is_p1") == 1)).sum().cast(pl.Int8).alias("p1_aggr_pf"),
        (pl.col("is_aggr") & pl.col("pf") & (pl.col("is_p1") == 0)).sum().cast(pl.Int8).alias("p2_aggr_pf"),
        pl.col("true_hu").sum().cast(pl.Int8).alias("hu_actions"),
        (pl.col("true_hu") & (pl.col("action") == "check")).sum().cast(pl.Int8).alias("hu_check"),
        (pl.col("true_hu") & (pl.col("action") == "call")).sum().cast(pl.Int8).alias("hu_call"),
        (pl.col("true_hu") & pl.col("is_aggr")).sum().cast(pl.Int8).alias("hu_aggr"),
        (pl.col("true_hu") & (pl.col("street_no") >= 2) & (pl.col("action") == "check")).sum().cast(pl.Int8).alias("hu_late_check"),
        (pl.col("true_hu") & (pl.col("action") == "fold")).sum().cast(pl.Int8).alias("hu_fold"),
        (pl.col("is_aggr") & (pl.col("amount_pot_ratio") >= 1.0)).sum().cast(pl.Int8).alias("pair_overbets"),
        pl.col("_post_loose_p").sum().cast(pl.Float32).alias("post_loose_partner"),
        pl.col("_fold_better_surp").sum().cast(pl.Float32).alias("fold_better_surprise"),
        pl.col("_junk_call_p").sum().cast(pl.Int8).alias("junk_call_partner_post"),
        pl.col("_fold_made_p").sum().cast(pl.Int8).alias("fold_made_partner"),
        pl.col("_made_check").sum().cast(pl.Int8).alias("made_check_sum"),
        (pl.col("true_hu") & pl.col("_made_check")).sum().cast(pl.Int8).alias("made_check_hu"),
    )
    responses = ac.filter((pl.col("to_call") > 0) & pl.col("last_aggr").is_not_null()).select(
        ["hand_id", pl.col("last_aggr").alias("member"), pl.col("player_id").alias("responder"), "action", "is_aggr", (pl.col("street_no") == 0).alias("pf")]
    )
    press = (
        members.join(responses, on=["hand_id", "member"], how="inner")
        .filter(pl.col("responder") != pl.col("partner"))
        .group_by(["pair_id", "hand_id"]).agg(
            (pl.col("action") == "fold").sum().cast(pl.Int8).alias("outsider_fold_to_pair"),
            (pl.col("action") == "call").sum().cast(pl.Int8).alias("outsider_call_to_pair"),
            pl.col("is_aggr").sum().cast(pl.Int8).alias("outsider_raise_to_pair"),
            ((pl.col("action") == "fold") & pl.col("pf")).sum().cast(pl.Int8).alias("outsider_fold_to_pair_pf"),
        )
    )
    out = (
        base.join(inter, on=["pair_id", "hand_id"], how="left").join(press, on=["pair_id", "hand_id"], how="left")
        .with_columns([pl.col(c).fill_null(0) for c in ACTION_FILL])
        .with_columns(
            ((pl.col("fold_to_partner") > 0) & (pl.col("loser_stronger_preflop") == 1)).cast(pl.Int8).alias("fold_stronger_to_partner"),
            ((pl.col("fold_to_partner") > 0) & (pl.col("fold_better_hand") == 1)).cast(pl.Int8).alias("fold_better_to_partner"),
            ((pl.col("loser_chen") >= 6) & (pl.col("winner_chen") >= 6) & (pl.col("raise_partner") == 0) & (pl.col("both_vpip") == 1)).cast(pl.Int8).alias("both_strong_no_raise"),
            (pl.col("call_partner") >= 2).cast(pl.Int8).alias("multi_call_partner"),
            ((pl.col("n_partners_aggr") == 2) & (pl.col("outsider_fold_to_pair") >= 1)).cast(pl.Int8).alias("squeeze"),
            ((pl.col("n_partners_aggr") == 2) & (pl.col("outsider_fold_to_pair") >= 1) & (pl.col("fold_to_partner") >= 1)).cast(pl.Int8).alias("squeeze_then_fold"),
            ((pl.col("raise_partner_pf") >= 1) & (pl.col("outsider_fold_to_pair_pf") >= 1)).cast(pl.Int8).alias("pf_squeeze"),
            ((pl.col("raise_partner_pf") >= 1) & (pl.col("outsider_fold_to_pair_pf") >= 1) & (pl.col("fold_to_partner") >= 1)).cast(pl.Int8).alias("squeeze_then_fold_pf"),
            ((pl.col("partner_won_other_lost") == 1) & (pl.col("call_partner") >= 1) & (pl.col("transfer_any") >= 10)).cast(pl.Int8).alias("dump"),
            ((pl.col("partner_won_other_lost") == 1) & (pl.col("call_partner") >= 1) & (pl.col("loser_beats_winner_final") == 1)).cast(pl.Int8).alias("dump_better_hand"),
            ((pl.col("both_showdown") == 1) & (pl.col("hu_check") >= 2) & (pl.col("pair_aggr") == 0)).cast(pl.Int8).alias("checkdown"),
            ((pl.col("both_showdown") == 1) & (pl.col("hu_late_check") >= 1) & (pl.max_horizontal("loser_chen", "winner_chen") >= 7)).cast(pl.Int8).alias("hu_checkdown_strong"),
            ((pl.col("hu_check") >= 1) & (pl.col("max_cat_final") >= 2) & (pl.col("pair_aggr") == 0)).cast(pl.Int8).alias("hu_check_two_pair_plus"),
            (pl.col("transfer_any") * pl.col("call_partner")).cast(pl.Float32).alias("transfer_x_call"),
            (pl.col("junk_call_partner_post") >= 1).cast(pl.Int8).alias("junk_call_after_partner_raise"),
            ((pl.col("p1_cat_final") >= 2) & (pl.col("p2_cat_final") >= 2) & (pl.col("pair_aggr") == 0) & (pl.col("both_showdown") == 1)).cast(pl.Int8).alias("both_passive_made"),
            (pl.col("fold_made_partner") >= 1).cast(pl.Int8).alias("fold_made_to_partner"),
        )
        .with_columns(
            (pl.col("transfer_any") / (pl.col("transfer_any").max().over("pair_id") + 1e-3)).cast(pl.Float32).alias("transfer_to_max"),
            (pl.col("net_gap") / (pl.col("net_gap").max().over("pair_id") + 1e-3)).cast(pl.Float32).alias("net_gap_to_max"),
            (pl.col("pot_bb") / (pl.col("pot_bb").max().over("pair_id") + 1e-3)).cast(pl.Float32).alias("pot_to_max"),
            (pl.col("outsider_fold_to_pair") / (pl.col("outsider_fold_to_pair").max().over("pair_id") + 1e-3)).cast(pl.Float32).alias("outsider_fold_to_max"),
            *[(pl.col(c).rank(method="average").over("pair_id") / pl.len().over("pair_id")).cast(pl.Float32).alias(f"{c}_prank") for c in PRANK_BASE],
        )
    )
    out.collect(engine="streaming").write_parquet(out_path)


def build_all_pair_hands(paths: Paths, tables, table_chunk, player_hands: Path) -> None:
    for k in range(N_TABLE_CHUNKS):
        chunk_tables = [t for t in tables if table_chunk[t] == k]
        build_pair_hand_features(
            pl.scan_parquet(paths.dev_pair_hands).filter(pl.col("table_id").is_in(chunk_tables)),
            paths.dev_hand_features(k),
            "development",
            paths,
            player_hands,
        )
        log(f"dev hand features chunk {k}: {pl.scan_parquet(paths.dev_hand_features(k)).select(pl.len()).collect().item():,} rows")
    build_pair_hand_features(pl.scan_parquet(paths.eval_pair_hands), paths.eval_hand_features, "evaluation", paths, player_hands)
    log(f"eval hand features: {pl.scan_parquet(paths.eval_hand_features).select(pl.len()).collect().item():,} rows; {len(HAND_FEATS)} hand features")


# ---------------------------------------------------------------------------
# Stage 7: hand-level models
# ---------------------------------------------------------------------------
@dataclass
class HandModels:
    dh: pd.DataFrame
    rank_models: dict
    sus_models: dict
    spec_models: dict
    rank_models_full: list
    sus_model_full: object
    spec_models_full: dict
    table_fold: dict
    thr95: float
    thr99: float
    sus95: float
    sus99: float
    oof_map5: float
    fam_map5: dict
    oof_map5_ev: float = 0.0
    fam_map5_ev: dict = field(default_factory=dict)


def train_hand_models(paths: Paths, dev_pairs: pl.DataFrame, evidence: pl.DataFrame, tables) -> tuple[pl.DataFrame, HandModels]:
    """Global evidence ranker + one specialist per family.

    Pair-level hs_* aggregates keep using the global score, so Pair AP does not
    move. MAP@5 slots are mixed later with pred_family.
    """
    labelled_ids = set(dev_pairs.filter(pl.col("is_labeled"))["pair_id"].to_list())
    lab = pl.concat([pl.scan_parquet(paths.dev_hand_features(k)).filter(pl.col("pair_id").is_in(list(labelled_ids))).collect() for k in range(N_TABLE_CHUNKS)])
    lab = lab.join(dev_pairs.select(["pair_id", "label", "behavior_family", "is_labeled"]), on="pair_id")
    lab = lab.join(evidence.select(["pair_id", "hand_id", pl.lit(True).alias("is_evidence")]), on=["pair_id", "hand_id"], how="left").with_columns(pl.col("is_evidence").fill_null(False))
    assert lab["is_evidence"].sum() == 1817, "not every evidence hand was reconstructed"
    dh = lab.to_pandas()
    dh[HAND_FEATS] = dh[HAND_FEATS].astype("float32")

    rng = np.random.RandomState(SEED)
    table_fold = {t: int(v) for t, v in zip(tables, rng.permutation(len(tables)) % N_FOLDS)}
    dh["fold"] = dh["table_id"].map(table_fold)
    dev_pairs = dev_pairs.with_columns(pl.col("table_id").replace_strict(table_fold, return_dtype=pl.Int8).alias("fold"))

    pos_mask = (dh["label"] == 1).to_numpy()
    dh["hand_score"] = 0.0
    dh["hand_sus"] = 0.0
    for col in SPEC_COLS:
        dh[col] = 0.0
    rank_models, sus_models, spec_models = {}, {}, {f: {} for f in range(N_FOLDS)}
    for fold in range(N_FOLDS):
        tr_pos = pos_mask & (dh["fold"] != fold).to_numpy()
        tr_all = (dh["fold"] != fold).to_numpy()
        te = (dh["fold"] == fold).to_numpy()
        X_te = dh.loc[te, HAND_FEATS]
        rank_models[fold] = _lgb_bag(dh.loc[tr_pos, HAND_FEATS], dh.loc[tr_pos, "is_evidence"].astype(int))
        sus_models[fold] = lgb.train(SUS_PARAMS, lgb.Dataset(dh.loc[tr_all, HAND_FEATS], dh.loc[tr_all, "is_evidence"].astype(int)), num_boost_round=HAND_ROUNDS)
        dh.loc[te, "hand_score"] = _predict_bag(rank_models[fold], X_te)
        dh.loc[te, "hand_sus"] = sus_models[fold].predict(X_te)
        for fam in TARGET_BEHAVIORS:
            tr_fam = tr_pos & (dh["behavior_family"] == fam).to_numpy()
            if int(tr_fam.sum()) < 200:
                spec_models[fold][fam] = rank_models[fold]
            else:
                spec_models[fold][fam] = _lgb_bag(dh.loc[tr_fam, HAND_FEATS], dh.loc[tr_fam, "is_evidence"].astype(int))
            dh.loc[te, f"hand_spec_{fam}"] = _predict_bag(spec_models[fold][fam], X_te)
        log(f"hand models fold {fold}: global MAP@5 on positive pairs = {map5_within_pairs(dh[te & pos_mask]):.4f}")

    dh["hand_score_ev"] = mix_evidence_scores(dh["hand_score"], _spec_matrix(dh), dh["behavior_family"])
    oof_map5, fam_map5 = report_map5(dh, "hand_score", "OOF evidence (global ranker)")
    report_map5(dh, "hand_score_ev", "OOF evidence (oracle family mix 0.60/0.40)")
    rank_full = _lgb_bag(dh.loc[pos_mask, HAND_FEATS], dh.loc[pos_mask, "is_evidence"].astype(int))
    sus_full = lgb.train(SUS_PARAMS, lgb.Dataset(dh[HAND_FEATS], dh["is_evidence"].astype(int)), num_boost_round=HAND_ROUNDS)
    spec_full = {}
    for fam in TARGET_BEHAVIORS:
        mask = pos_mask & (dh["behavior_family"] == fam).to_numpy()
        spec_full[fam] = _lgb_bag(dh.loc[mask, HAND_FEATS], dh.loc[mask, "is_evidence"].astype(int)) if int(mask.sum()) >= 200 else rank_full
    neg_scores = dh.loc[(dh["label"] == 0).to_numpy(), "hand_score"].to_numpy()
    neg_sus = dh.loc[(dh["label"] == 0).to_numpy(), "hand_sus"].to_numpy()
    imp = pd.Series(np.mean([m.feature_importance("gain") for m in rank_full], axis=0), index=HAND_FEATS).sort_values(ascending=False)
    print("top evidence-ranker features (gain):\n", imp.head(20).round(0).to_string())
    models = HandModels(
        dh=dh, rank_models=rank_models, sus_models=sus_models, spec_models=spec_models,
        rank_models_full=rank_full, sus_model_full=sus_full, spec_models_full=spec_full,
        table_fold=table_fold, thr95=float(np.quantile(neg_scores, 0.95)), thr99=float(np.quantile(neg_scores, 0.99)),
        sus95=float(np.quantile(neg_sus, 0.95)), sus99=float(np.quantile(neg_sus, 0.99)),
        oof_map5=oof_map5, fam_map5=fam_map5,
    )
    return dev_pairs, models


# ---------------------------------------------------------------------------
# Stage 8: pair aggregation + two-step PU
# ---------------------------------------------------------------------------
def aggregate_pairs(hf: pl.DataFrame, baselines_phase: pl.DataFrame, pairs: pl.DataFrame, hand: HandModels) -> pd.DataFrame:
    """Collapse pair-hands to one row per pair: rates, tails, partner-vs-field contrasts, table ranks."""
    cols = (
        ["pair_id", "hand_id", "table_id", "hand_idx", "player_1", "player_2", "hand_score", "hand_sus",
         "loser_is_p1", "signed_net_diff", "transfer_1_to_2", "transfer_2_to_1", "p1_call_partner", "p1_fold_to_partner"]
        + [f"p{i}_{c}" for i in (1, 2) for c in CONTRAST]
        + sorted(set(AGG_MEAN + AGG_MAX + AGG_TOP3 + AGG_SUM))
    )
    lf = hf.select(cols).lazy()
    aggs = [pl.len().alias("n_shared"), pl.col("table_id").first(), pl.col("player_1").first(), pl.col("player_2").first()]
    aggs += [pl.col(c).mean().alias(f"{c}_mean") for c in AGG_MEAN]
    aggs += [pl.col(c).max().alias(f"{c}_max") for c in AGG_MAX]
    aggs += [pl.col(c).top_k(3).mean().alias(f"{c}_top3") for c in AGG_TOP3]
    aggs += [pl.col(c).sum().alias(f"{c}_sum") for c in AGG_SUM]
    aggs += [pl.col(f"p{i}_{c}").cast(pl.Float32).mean().alias(f"_p{i}_{c}") for i in (1, 2) for c in CONTRAST]
    aggs += [
        pl.col("hand_score").mean().alias("hs_mean"), pl.col("hand_score").max().alias("hs_max"),
        pl.col("hand_score").top_k(3).mean().alias("hs_top3"), pl.col("hand_score").top_k(5).mean().alias("hs_top5"),
        pl.col("hand_score").quantile(0.9).alias("hs_p90"),
        (pl.col("hand_score") > hand.thr95).sum().alias("hs_n95"),
        (pl.col("hand_score") > hand.thr99).sum().alias("hs_n99"),
        (pl.col("hand_score") > hand.thr95).mean().alias("hs_f95"),
        pl.col("hand_sus").mean().alias("sus_mean"), pl.col("hand_sus").max().alias("sus_max"),
        pl.col("hand_sus").top_k(3).mean().alias("sus_top3"), pl.col("hand_sus").top_k(5).mean().alias("sus_top5"),
        pl.col("hand_sus").quantile(0.9).alias("sus_p90"),
        (pl.col("hand_sus") > hand.sus95).sum().alias("sus_n95"),
        (pl.col("hand_sus") > hand.sus99).sum().alias("sus_n99"),
        (pl.col("hand_sus") > hand.sus95).mean().alias("sus_f95"),
        pl.col("transfer_1_to_2").sum().alias("_t12"), pl.col("transfer_2_to_1").sum().alias("_t21"),
        pl.col("signed_net_diff").sum().alias("_snd"), pl.col("signed_net_diff").abs().sum().alias("_asnd"),
        pl.col("p1_call_partner").sum().alias("_p1_cp"), pl.col("call_partner").sum().alias("_cp"),
        pl.col("p1_fold_to_partner").sum().alias("_p1_fp"), pl.col("fold_to_partner").sum().alias("_fp"),
        pl.col("loser_is_p1").sort_by("hand_score", descending=True).head(5).mean().alias("_top5_loser_p1"),
        pl.col("loser_is_p1").sort_by("hand_sus", descending=True).head(5).mean().alias("_top5s_loser_p1"),
        pl.col("loser_is_p1").sort_by("transfer_any", descending=True).head(5).mean().alias("_top5t_loser_p1"),
    ]
    pf = lf.group_by("pair_id").agg(aggs)
    burst = (
        lf.with_columns(
            ((pl.col("hand_idx").cast(pl.Float32) - pl.col("hand_idx").min().over("pair_id"))
             / (pl.col("hand_idx").max().over("pair_id") - pl.col("hand_idx").min().over("pair_id") + 1) * 8)
            .floor().clip(0, 7).cast(pl.Int8).alias("_bin")
        )
        .group_by(["pair_id", "_bin"]).agg(
            pl.col("hand_score").mean().alias("_b"),
            pl.col("hand_sus").mean().alias("_bs"),
            pl.col("transfer_any").mean().alias("_bt"),
        )
        .group_by("pair_id").agg(
            pl.col("_b").max().alias("hs_burst_max"), (pl.col("_b").max() - pl.col("_b").mean()).alias("hs_burst_excess"),
            pl.col("_bs").max().alias("sus_burst_max"), (pl.col("_bs").max() - pl.col("_bs").mean()).alias("sus_burst_excess"),
            pl.col("_bt").max().alias("transfer_burst_max"),
        )
    )
    pf = pf.join(burst, on="pair_id").collect()
    for pfx, key in (("p1", "player_1"), ("p2", "player_2")):
        pf = pf.join(
            baselines_phase.rename({c: f"{pfx}_{c}" for c in baselines_phase.columns if c != "player_id"}).rename({"player_id": key}),
            on=key, how="left",
        )
    exprs = []
    for c in CONTRAST:
        for i in (1, 2):
            field = (pl.col(f"p{i}_base_{c}") * pl.col(f"p{i}_base_hands") - pl.col(f"_p{i}_{c}") * pl.col("n_shared")) / (
                pl.col(f"p{i}_base_hands") - pl.col("n_shared")
            ).clip(lower_bound=1)
            exprs.append(((pl.col(f"_p{i}_{c}") - field) * pl.col("n_shared") / (pl.col("n_shared") + SHRINK_N)).cast(pl.Float32).alias(f"p{i}_{c}_pf"))
    pf = pf.with_columns(exprs).with_columns(
        *[(pl.col(f"p1_{c}_pf") + pl.col(f"p2_{c}_pf")).alias(f"{c}_pf_sum") for c in CONTRAST],
        *[(pl.col(f"p1_{c}_pf") - pl.col(f"p2_{c}_pf")).abs().alias(f"{c}_pf_gap") for c in CONTRAST],
        *[pl.min_horizontal(f"p1_{c}_pf", f"p2_{c}_pf").alias(f"{c}_pf_min") for c in CONTRAST],
        ((pl.col("_t12") - pl.col("_t21")).abs() / (pl.col("_t12") + pl.col("_t21") + 1.0)).alias("transfer_imbalance"),
        ((pl.col("_t12") + pl.col("_t21")) / pl.col("n_shared")).alias("transfer_rate"),
        pl.max_horizontal("_t12", "_t21").alias("transfer_dominant"),
        (pl.col("_snd").abs() / (pl.col("_asnd") + 1.0)).alias("direction_consistency"),
        pl.max_horizontal(pl.col("_top5_loser_p1"), 1 - pl.col("_top5_loser_p1")).alias("top5_same_loser"),
        pl.max_horizontal(pl.col("_top5s_loser_p1"), 1 - pl.col("_top5s_loser_p1")).alias("top5s_same_loser"),
        pl.max_horizontal(pl.col("_top5t_loser_p1"), 1 - pl.col("_top5t_loser_p1")).alias("top5t_same_loser"),
        (pl.max_horizontal(pl.col("_p1_cp"), pl.col("_cp") - pl.col("_p1_cp")) / (pl.col("_cp") + 1.0)).alias("call_partner_asym"),
        (pl.max_horizontal(pl.col("_p1_fp"), pl.col("_fp") - pl.col("_p1_fp")) / (pl.col("_fp") + 1.0)).alias("fold_partner_asym"),
        (pl.col("_p1_net_bb") - pl.col("p1_base_net_bb")).alias("p1_net_resid"),
        (pl.col("_p2_net_bb") - pl.col("p2_base_net_bb")).alias("p2_net_resid"),
    ).with_columns(
        (pl.col("p1_net_resid") - pl.col("p2_net_resid")).abs().alias("net_resid_gap"),
        pl.min_horizontal("p1_net_resid", "p2_net_resid").alias("net_resid_min"),
        pl.max_horizontal("p1_net_resid", "p2_net_resid").alias("net_resid_max"),
    )
    drop = [c for c in pf.columns if c.startswith("_") or c.startswith("p1_base_") or c.startswith("p2_base_")]
    pf = pf.drop(drop)
    pf = pf.with_columns([
        (pl.col(c).rank(method="average").over("table_id") / pl.len().over("table_id")).cast(pl.Float32).alias(f"{c}_trank")
        for c in TRANK_COLS if c in pf.columns
    ])
    keep = [c for c in pairs.columns if c in ("pair_id", "label", "behavior_family", "is_labeled", "fold", "shared_hands", "chunk")]
    return pf.join(pairs.select(keep), on="pair_id", how="left").to_pandas()


def score_dev_chunks(paths: Paths, hand: HandModels, baselines, dev_pairs: pl.DataFrame) -> pd.DataFrame:
    _drop_if_stale(paths.dev_pair_features, PAIR_CACHE_COLS)
    if paths.dev_pair_features.exists():
        return pd.read_parquet(paths.dev_pair_features)
    parts = []
    base_dev = baselines.filter(pl.col("phase") == "development").drop("phase")
    for k in range(N_TABLE_CHUNKS):
        hf = pl.read_parquet(paths.dev_hand_features(k))
        fold_of_row = hf["table_id"].replace_strict(hand.table_fold, return_dtype=pl.Int8).to_numpy()
        X = hf.select(HAND_FEATS).to_pandas().astype("float32")
        hs = np.zeros(hf.height, np.float32)
        su = np.zeros(hf.height, np.float32)
        for fold in range(N_FOLDS):
            mask = fold_of_row == fold
            if mask.any():
                hs[mask] = np.mean([m.predict(X[mask]) for m in hand.rank_models[fold]], axis=0)
                su[mask] = hand.sus_models[fold].predict(X[mask])
        hf = hf.with_columns(pl.Series("hand_score", hs), pl.Series("hand_sus", su))
        parts.append(aggregate_pairs(hf, base_dev, dev_pairs, hand))
        log(f"dev chunk {k}: {hf.height:,} pair-hands scored OOF -> {len(parts[-1]):,} pairs")
        del hf, X
        gc.collect()
    dev_pf = pd.concat(parts, ignore_index=True)
    dev_pf.to_parquet(paths.dev_pair_features, index=False)
    return dev_pf


def predict_family(fam_probs: dict, fam_prior: dict) -> np.ndarray:
    mat = np.column_stack([fam_probs[fam] / fam_prior[fam] for fam in TARGET_BEHAVIORS])
    return np.array(TARGET_BEHAVIORS)[mat.argmax(axis=1)]


@dataclass
class PairModels:
    pair_models: list
    fam_models: dict
    pair_feats: list
    oof: np.ndarray
    behavior_rho: float
    labelled_ap: float
    pu_ap: float
    fam_prior: dict
    host_components: dict


def fit_pair_cv(dev_pf, X, y, labm, w_fit, folds, tag, seeds=PAIR_SEEDS, rounds=PAIR_ROUNDS, with_family=True):
    oof = np.zeros(len(dev_pf))
    models = []
    fam_models = {fam: [] for fam in TARGET_BEHAVIORS}
    oof_fam = {fam: np.zeros(len(dev_pf)) for fam in TARGET_BEHAVIORS}
    for fold in range(N_FOLDS):
        tr, te = folds != fold, folds == fold
        tr_w = tr & (w_fit > 0)
        for sd in seeds:
            model = lgb.train({**PAIR_PARAMS, "seed": sd}, lgb.Dataset(X[tr_w], y[tr_w], weight=w_fit[tr_w]), num_boost_round=rounds)
            oof[te] += model.predict(X[te]) / len(seeds)
            models.append(model)
        if with_family:
            tr_f = tr_w & ~((y == 1) & ~labm)
            for fam in TARGET_BEHAVIORS:
                yf = (dev_pf["behavior_family"] == fam).to_numpy().astype(int)
                mf = lgb.train({**PAIR_PARAMS, "num_leaves": 7}, lgb.Dataset(X[tr_f], yf[tr_f], weight=w_fit[tr_f]), num_boost_round=FAMILY_ROUNDS)
                oof_fam[fam][te] = mf.predict(X[te])
                fam_models[fam].append(mf)
        lab = labm & te
        log(f"{tag} fold {fold}: labelled AP={average_precision_score(y[lab], oof[lab]):.4f} | eval-mirrored AP (all pairs)={average_precision_score(y[te], oof[te]):.4f}")
    return oof, models, oof_fam, fam_models


def train_pair_models(dev_pf: pd.DataFrame, hand: HandModels, evidence: pl.DataFrame) -> PairModels:
    """Stage 1 treats unlabelled pairs as weak negatives. Stage 2 relabels the ones that look like colluders."""
    pair_feats = [c for c in dev_pf.columns if c not in PAIR_META and not c.startswith("sus_") and c != "top5s_same_loser"]
    log(f"dev pair features: {dev_pf.shape}; {len(pair_feats)} features")
    y = dev_pf["label"].fillna(0).astype(int).to_numpy()
    labm = dev_pf["is_labeled"].to_numpy()
    w_train = np.where(labm, np.where(y == 1, POS_WEIGHT, NEG_WEIGHT), PU_WEIGHT)
    folds = dev_pf["fold"].to_numpy()
    X = dev_pf[pair_feats].astype("float32")

    oof1, _, _, _ = fit_pair_cv(dev_pf, X, y, labm, w_train, folds, "stage-1 pair model", seeds=PAIR_SEEDS[:1], rounds=PAIR_ROUNDS, with_family=False)
    pos_oof = oof1[labm & (y == 1)]
    thr_ambig, thr_pseudo = np.quantile(pos_oof, 0.05), np.quantile(pos_oof, 0.50)
    unl = ~labm
    ambiguous = unl & (oof1 >= thr_ambig) & (oof1 < thr_pseudo)
    pseudo = unl & (oof1 >= thr_pseudo)
    print(
        f"two-step PU: thresholds ambiguous>={thr_ambig:.4f} pseudo-positive>={thr_pseudo:.4f} | "
        f"unlabelled: {int(pseudo.sum())} pseudo-positives, {int(ambiguous.sum())} excluded, "
        f"{int((unl & ~ambiguous & ~pseudo).sum())} reliable negatives"
    )
    y2 = np.where(pseudo, 1, y)
    w2 = np.where(labm, np.where(y == 1, POS_WEIGHT, NEG_WEIGHT), np.where(pseudo, PSEUDO_POS_WEIGHT, np.where(ambiguous, 0.0, PU_WEIGHT)))
    oof, pair_models, oof_fam, fam_models = fit_pair_cv(dev_pf, X, y2, labm, w2, folds, "stage-2 pair model")
    clean = ~ambiguous & ~pseudo
    print(f"stage-1 eval-mirrored AP {average_precision_score(y, oof1):.4f} -> stage-2 {average_precision_score(y, oof):.4f} (all unlabelled as negatives, pessimistic)")
    print(f"stage-2 AP on the population without the {int((ambiguous | pseudo).sum())} colluder-like unlabelled pairs: {average_precision_score(y[clean], oof[clean]):.4f}")
    print(f"stage-2 AP if pseudo-positives are counted as positives: {average_precision_score(y2[~ambiguous], oof[~ambiguous]):.4f}")

    labelled_ap = average_precision_score(y[labm], oof[labm])
    pu_ap = average_precision_score(y, oof)
    print(f"\nOOF labelled Pair AP (positives vs confirmed negatives, OPTIMISTIC): {labelled_ap:.4f}")
    print(f"OOF eval-mirrored Pair AP (positives vs ALL eligible pairs, tracks LB): {pu_ap:.4f}")
    for fam in TARGET_BEHAVIORS:
        sel = (dev_pf["behavior_family"] == fam).to_numpy() | (y == 0)
        print(f"   {fam:22s} eval-mirrored AP = {average_precision_score(y[sel], oof[sel]):.4f}")

    fam_prior = {fam: (dev_pf["behavior_family"] == fam).sum() for fam in TARGET_BEHAVIORS}
    dev_pf = dev_pf.copy()
    dev_pf["pred_family"] = predict_family(oof_fam, fam_prior)
    pos = dev_pf["label"] == 1
    print(f"family accuracy on positives: {(dev_pf.loc[pos, 'pred_family'] == dev_pf.loc[pos, 'behavior_family']).mean():.3f}")

    pred_map = dev_pf.set_index("pair_id")["pred_family"]
    hand.dh["pred_family"] = hand.dh["pair_id"].map(pred_map)
    w_pair = family_weight_matrix(oof_fam, fam_prior)
    w_map = {pid: w_pair[i] for i, pid in enumerate(dev_pf["pair_id"].to_numpy())}
    w_hand = np.stack([w_map[p] for p in hand.dh["pair_id"].to_numpy()])
    spec = _spec_matrix(hand.dh)
    hard = mix_evidence_scores(hand.dh["hand_score"], spec, hand.dh["pred_family"])
    soft = mix_evidence_soft(hand.dh["hand_score"], spec, w_hand)
    report_map5(hand.dh.assign(hand_score_ev=hard), "hand_score_ev", "OOF evidence (hard pred_family mix 0.60/0.40)")
    hand.dh["hand_score_ev"] = soft
    hand.oof_map5_ev, hand.fam_map5_ev = report_map5(
        hand.dh, "hand_score_ev", "OOF evidence (soft family mix 0.60/0.40) — this is the leaderboard mix"
    )

    top5 = (
        hand.dh.sort_values(["pair_id", "hand_score_ev", "pot_bb", "hand_id"], ascending=[True, False, False, True])
        .groupby("pair_id")["hand_id"].apply(lambda s: list(s.head(5)))
    )
    sol = pd.DataFrame({"pair_id": dev_pf["pair_id"], "risk_score": y, "predicted_behavior": np.where(y == 1, dev_pf["behavior_family"], "none")})
    ev_map = evidence.to_pandas().groupby("pair_id")["hand_id"].apply(list)
    for i, col in enumerate(EVIDENCE_COLUMNS):
        sol[col] = [(ev_map[p][i] if (p in ev_map.index and i < len(ev_map[p])) else NO_EVIDENCE) for p in sol["pair_id"]]
    sub_dev = pd.DataFrame({"pair_id": dev_pf["pair_id"], "risk_score": oof, "predicted_behavior": dev_pf["pred_family"]})
    for i, col in enumerate(EVIDENCE_COLUMNS):
        sub_dev[col] = [(top5[p][i] if (p in top5.index and i < len(top5[p])) else NO_EVIDENCE) for p in sub_dev["pair_id"]]
    comp = host_score(sol, sub_dev, return_components=True)
    print("host metric on the full development population (family for every pair):", json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in comp.items()}, default=str))

    rank_pct = pd.Series(oof).rank(ascending=False, method="first").to_numpy() / len(oof)
    none_scores = {}
    for rho in NONE_GRID:
        sd = sub_dev.copy()
        sd["predicted_behavior"] = np.where(rank_pct <= rho, dev_pf["pred_family"], "none")
        none_scores[rho] = host_score(sol, sd, return_components=True)
        print(f"  rho={rho:<7} final={none_scores[rho]['final']:.4f} behavior_map={none_scores[rho]['behavior_map']:.4f}")
    best = max(v["final"] for v in none_scores.values())
    behavior_rho = max(r for r in NONE_GRID if r <= 0.05 and none_scores[r]["final"] >= best - 0.0005)
    comp = none_scores[behavior_rho]
    print(f"selected behaviour rate rho={behavior_rho} -> host metric on the development population: {json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in comp.items()}, default=str)}")
    imp = pd.Series(np.mean([m.feature_importance("gain") for m in pair_models], axis=0), index=pair_feats).sort_values(ascending=False)
    print("top pair features (gain):\n", imp.head(25).round(0).to_string())
    return PairModels(
        pair_models=pair_models, fam_models=fam_models, pair_feats=pair_feats, oof=oof,
        behavior_rho=behavior_rho, labelled_ap=labelled_ap, pu_ap=pu_ap, fam_prior=fam_prior, host_components=comp,
    )


# ---------------------------------------------------------------------------
# Stage 9: evaluation inference + submission
# ---------------------------------------------------------------------------
def score_evaluation(paths: Paths, hand: HandModels, pair: PairModels, baselines, eval_prep: pl.DataFrame):
    scored_cols = {"hand_score", "hand_sus", *SPEC_COLS, *HAND_CACHE_COLS}
    have = set(pl.scan_parquet(paths.eval_hand_scored).collect_schema().names()) if paths.eval_hand_scored.exists() else set()
    if scored_cols - have:
        if paths.eval_hand_scored.exists():
            paths.eval_hand_scored.unlink()
        parts = []
        lf = pl.scan_parquet(paths.eval_hand_features)
        n = lf.select(pl.len()).collect().item()
        for start in range(0, n, EVAL_SCORE_CHUNK):
            chunk = lf.slice(start, EVAL_SCORE_CHUNK).collect()
            Xc = chunk.select(HAND_FEATS).to_pandas().astype("float32")
            s1 = _predict_bag(hand.rank_models_full, Xc)
            s2 = hand.sus_model_full.predict(Xc)
            spec = {fam: _predict_bag(hand.spec_models_full[fam], Xc).astype(np.float32) for fam in TARGET_BEHAVIORS}
            parts.append(
                chunk.with_columns(
                    pl.Series("hand_score", s1, dtype=pl.Float32),
                    pl.Series("hand_sus", s2, dtype=pl.Float32),
                    *[pl.Series(f"hand_spec_{fam}", spec[fam], dtype=pl.Float32) for fam in TARGET_BEHAVIORS],
                )
            )
            log(f"scored eval hands {min(start + EVAL_SCORE_CHUNK, n):,}/{n:,}")
        pl.concat(parts).write_parquet(paths.eval_hand_scored)
        del parts, chunk, Xc
        gc.collect()
    eh = pl.read_parquet(paths.eval_hand_scored)
    if "predicted_behavior" in eh.columns:
        eh = eh.drop("predicted_behavior")
    eval_pf = aggregate_pairs(eh, baselines.filter(pl.col("phase") == "evaluation").drop("phase"), eval_prep, hand)
    assert len(eval_pf) == 112_540
    eval_pf.to_parquet(paths.work / "eval_pair_features.parquet", index=False)
    Xe = eval_pf[pair.pair_feats].astype("float32")
    risk = np.mean([m.predict(Xe) for m in pair.pair_models], axis=0)
    fam_probs = {fam: np.mean([m.predict(Xe) for m in pair.fam_models[fam]], axis=0) for fam in TARGET_BEHAVIORS}
    eval_pf["risk_score"] = risk
    rank_pct = pd.Series(risk).rank(ascending=False, method="first").to_numpy() / len(risk)
    eval_pf["predicted_behavior"] = np.where(rank_pct <= pair.behavior_rho, predict_family(fam_probs, pair.fam_prior), "none")
    eh = eh.join(pl.from_pandas(eval_pf[["pair_id", "predicted_behavior"]]), on="pair_id", how="left")
    w_pair = family_weight_matrix(fam_probs, pair.fam_prior)
    w_df = pl.from_pandas(pd.DataFrame(
        w_pair, columns=[f"_w_{fam}" for fam in TARGET_BEHAVIORS]
    ).assign(pair_id=eval_pf["pair_id"].to_numpy()))
    eh = eh.join(w_df, on="pair_id", how="left")
    weights = eh.select([f"_w_{fam}" for fam in TARGET_BEHAVIORS]).to_numpy()
    active = (eh["predicted_behavior"] != "none").to_numpy()
    ev = mix_evidence_soft(eh["hand_score"].to_numpy(), _spec_matrix(eh), weights, active)
    eh = eh.with_columns(pl.Series("hand_score_ev", ev)).drop([f"_w_{fam}" for fam in TARGET_BEHAVIORS])
    top = (
        eh.select(["pair_id", "hand_id", "hand_score_ev", "pot_bb"])
        .sort(["pair_id", "hand_score_ev", "pot_bb", "hand_id"], descending=[False, True, True, False])
        .group_by("pair_id", maintain_order=True).head(5)
        .with_columns(pl.int_range(0, pl.len()).over("pair_id").alias("rank"))
        .pivot(on="rank", index="pair_id", values="hand_id")
    )
    top = top.rename({str(i): f"evidence_hand_{i+1}" for i in range(5)}) if "0" in top.columns else top.rename({i: f"evidence_hand_{i+1}" for i in range(5)})
    log(f"evidence lists built for {top.height:,} pairs")
    return eval_pf, eh, top


def write_submission(paths: Paths, eval_pf, eh, top, sample_sub, pair: PairModels, hand: HandModels) -> None:
    sub = eval_pf[["pair_id", "risk_score", "predicted_behavior"]].merge(top.to_pandas(), on="pair_id", how="left")
    sub["risk_score"] = sub["risk_score"].clip(0, 1).round(9)
    sec = eval_pf.set_index("pair_id").loc[sub["pair_id"], "hs_max"].to_numpy()
    order = pd.DataFrame({"r": sub["risk_score"], "sec": sec}).sort_values(["r", "sec"], ascending=[True, False], kind="mergesort")
    dup_rank = order.groupby("r").cumcount().reindex(sub.index)
    sub["risk_score"] = np.clip(sub["risk_score"] + 1e-13 * dup_rank.to_numpy(), 0, 1)
    assert sub["risk_score"].is_unique, "risk_score still has ties"
    for col in EVIDENCE_COLUMNS:
        sub[col] = sub[col].fillna(NO_EVIDENCE).astype(str)
    sub = sample_sub.select("pair_id").to_pandas().merge(sub, on="pair_id", how="left")

    assert len(sub) == 112_540 and sub["pair_id"].is_unique
    assert sub.isna().sum().sum() == 0
    assert sub["risk_score"].between(0, 1).all()
    assert set(sub["predicted_behavior"]) <= set(TARGET_BEHAVIORS) | {"none"}
    assert (sub[list(EVIDENCE_COLUMNS)] == NO_EVIDENCE).sum().sum() == 0, "every pair has >= 38 shared hands; no NO_EVIDENCE expected"
    ev_long = sub.melt(id_vars="pair_id", value_vars=list(EVIDENCE_COLUMNS), value_name="hand_id")
    assert not ev_long.duplicated(["pair_id", "hand_id"]).any(), "duplicate evidence hand within a pair"
    shared_keys = eh.select(["pair_id", "hand_id"]).to_pandas()
    assert ev_long.merge(shared_keys, on=["pair_id", "hand_id"], how="left", indicator=True)["_merge"].eq("both").all(), "evidence hand not a shared evaluation hand"
    print("risk ties:", sub["risk_score"].duplicated().sum(), "| behaviours:", sub["predicted_behavior"].value_counts().to_dict())

    fake = sub[["pair_id"]].copy()
    rs = np.random.RandomState(0)
    fake["risk_score"] = (rs.rand(len(fake)) < 0.003).astype(int)
    fake["predicted_behavior"] = np.where(fake["risk_score"] == 1, rs.choice(TARGET_BEHAVIORS, len(fake)), "none")
    for col in EVIDENCE_COLUMNS:
        fake[col] = np.where(fake["risk_score"] == 1, sub[col], NO_EVIDENCE)
    print("host metric accepts the file; score vs a random fake solution:", round(host_score(fake, sub), 4))

    sub.to_csv(paths.sub, index=False)
    chk = pd.read_csv(paths.sub, dtype={c: str for c in EVIDENCE_COLUMNS})
    assert chk.shape == (112_540, 8)
    log(f"wrote {paths.sub} ({paths.sub.stat().st_size/1e6:.1f} MB)")
    print(chk.head(3).to_string())
    print(json.dumps({
        "behaviour_rho": pair.behavior_rho,
        "oof_labelled_pair_ap": round(pair.labelled_ap, 4),
        "oof_eval_mirrored_pair_ap": round(pair.pu_ap, 4),
        "oof_evidence_map5_global": round(hand.oof_map5, 4),
        "oof_evidence_map5_pred_family_mix": round(hand.oof_map5_ev, 4),
        "evidence_map5_by_family_global": {k: round(v, 4) for k, v in hand.fam_map5.items()},
        "evidence_map5_by_family_pred_mix": {k: round(v, 4) for k, v in (hand.fam_map5_ev or {}).items()},
        "dev_population_host_metric": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in pair.host_components.items()},
    }, indent=2, default=str))


def main() -> None:
    paths = resolve_paths()
    log(f"DATA_DIR={paths.data}  WORK_DIR={paths.work}")
    labels, evidence, eval_pairs, sample_sub, hands, n_dev, positives = load_inputs(paths)

    build_action_context(paths, hands)
    log(f"action context rows: {pl.scan_parquet(paths.action_context).select(pl.len()).collect().item():,}")
    build_player_hands(paths, hands)
    player_hands = attach_policy(paths)
    build_player_strength(paths)
    log(f"player strength rows: {pl.scan_parquet(paths.player_strength).select(pl.len()).collect().item():,}")
    player_hands = attach_postflop_policy(paths, player_hands)
    baselines = player_baselines(player_hands)
    log(f"player-hand rows: {pl.scan_parquet(player_hands).select(pl.len()).collect().item():,}; baselines: {baselines.height:,}")

    dev_pairs, eval_prep, tables, table_chunk = build_pair_index(paths, labels, eval_pairs, n_dev, positives, player_hands)
    build_all_pair_hands(paths, tables, table_chunk, player_hands)
    dev_pairs, hand = train_hand_models(paths, dev_pairs, evidence, tables)
    dev_pf = score_dev_chunks(paths, hand, baselines, dev_pairs)
    pair = train_pair_models(dev_pf, hand, evidence)
    eval_pf, eh, top = score_evaluation(paths, hand, pair, baselines, eval_prep)
    write_submission(paths, eval_pf, eh, top, sample_sub, pair, hand)


if __name__ == "__main__":
    main()
